# Suite DIA — The Python, Matlab, Excel, Latex, Ccpp and SystemVerilog dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules (modeled over JavaScript values here);
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. Latex writes notation, and has no evaluator. Ccpp evaluates by C's
rules over its arithmetic types, and SystemVerilog by IEEE 1800's over 4-state vectors and reals. What every dialect shares is suite FRM; the Basic
dialect is suites EXP and EVL. Reading Python source (`parse`) and NumPy are the Python implementation's only.

In [ ]:
import * as Basic from "@mbse/expressions/Dialects/Basic";
import * as Ccpp from "@mbse/expressions/Dialects/Ccpp";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python from "@mbse/expressions/Dialects/Python";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import { Domains as D, Errors as FE, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Errors, Proxies, Repr, Schemas as S } from "@mbse/schemas/Framework";

const { AttributeError, KeyError, ValueError } = Errors;
const { ImportError, IndexError, NameError, OverflowError, ZeroDivisionError } = FE;
import { assert, raises, same, text } from "./support.js";

const [P, M, X] = [Python.Expressions, Matlab.Expressions, Excel.Expressions];
const [PV, MV, XV] = [Python.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [PD, MD, XD] = [Python.Domains, Matlab.Domains, Excel.Domains];
const [C, CV, CD, BD] = [Ccpp.Expressions, Ccpp.Evaluators, Ccpp.Domains, Basic.Domains];
const [SV, SVV, SVD] = [SystemVerilog.Expressions, SystemVerilog.Evaluators, SystemVerilog.Domains];
const kindOf = (dialect: F.Declared, tag: string) => dialect.kinds().get(tag) as any;

const Person = new S.OfObject.Builder().ref().properties(text("name"), text("age", BigInt), text("email")).create();
Proxies.register("Person", Person);
const B = (Proxies.Builders as any).Person;
const ann = B().name("Ann").age(30n).email("ann@example.com").create();
const bob = B().name("Bob").age(70n).create(); // no email
const [PText, MText, XText, LText, CText, SVText] = [Python.Text, Matlab.Text, Excel.Text, Latex.Text, Ccpp.Text, SystemVerilog.Text];

## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
{
  const [t, a, b, c, d, x, y, z] = [..."tabcdxyz"].map((n) => P.name(n));
  for (const [built, source] of [ // written with constructors, as source
    [P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
      "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18n),
      P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), "np"),
    "import numpy as np\nfrom numpy.ma import getmaskarray\n"
      + "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"],
    [P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1n), P.unaryop("not", P.attribute(a, "c")))),
      "(lambda a: a.b > 1 and not a.c)(t)"],
    [P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2n))), "x if a < b else -y ** 2"],
    [P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"],
    [P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"],
    [P.binop("-", P.binop("-", P.binop("**", 2n, P.unaryop("-", 1n)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"],
    [P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"],
    [P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"],
    [P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"],
    [P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1n), new Uint8Array([0]), true, "it's"),
      "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"],
    [P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1n, "g")), "c")),
      "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"],
    [P.call("all", P.generator("p", P.attribute(t, "ports"), P.compare(">", P.attribute(P.name("p"), "pin"), 0n))),
      "all(p.pin > 0 for p in t.ports)"],
    [P.call("sum", P.generator("p", x, 1n, P.compare("in", P.name("p"), y), P.compare("not in", P.name("p"), z))),
      "sum(1 for p in x if p in y if p not in z)"],
    [P.binop("+", P.index(x, P.binop("-", a, 1n)), P.call("len", P.call("set", x))), "x[a - 1] + len(set(x))"],
    [P.call("max", P.generator("v", P.ifexp(a, x, y), P.ifexp(b, P.name("v"), z), P.boolop("or", a, b)), 1n),
      "max((v if b else z for v in (x if a else y) if a or b), 1)"],
  ] as [F.Term, string][]) {
    assert(PText.ToText(built) === source, PText.ToText(built));
  }
  assert(!("FromText" in PText)); // reading Python source needs Python's parser: the Python implementation's only
  assert(same([NaN, Infinity, -Infinity, -2n].map((v) => PText.ToText(P.constant(v))), [
    "float('nan')", "float('inf')", "-float('inf')", "-2"]));
  assert(PText.ToText(P.binop("**", -2n, 2n)) === "(-2) ** 2" && PText.ToText(P.unaryop("-", P.binop("**", 2n, 2n))) === "-2 ** 2");
  assert(PText.ToText(P.call("np.ma.getmaskarray", P.name("x"))) === "np.ma.getmaskarray(x)"); // a dotted function
  assert(PText.ToText(P.attribute(P.binop("+", 1n, 2n), "real")) === "(1 + 2).real");
  const counted = P.generator("p", P.name("ps"), P.name("p"), P.name("q")); // p is bound in the element and the conditions only
  assert(same([...SY.free(counted)].sort(), ["ps", "q"]) && counted.validate({ bound: ["ps", "q"] }).length === 0);
  assert(same(P.generator("1p", P.name("ps"), 1n).validate({ bound: ["ps"] }), ["a generator's name must be an identifier, got '1p'"]));
  assert(same(P.generator("p", P.name("p"), 1n).validate(), ["iterable: name 'p' is not bound"]));
  raises(ValueError, () => PText.ToText(P.binop("+", 1n, P.import_("math", 2n))), "an import can only enclose the whole expression");
  assert(same(P.import_("numpy.ma", 1n).binds(), ["numpy"]) && same(P.importfrom("m", "n", 1n, "k").binds(), ["k"]));
  assert(same(P.import_("not a module", 1n).validate(), ["an import's module must be a dotted name, got 'not a module'"]));
  assert(same(P.import_("m", 1n, "a b").validate(), ["an import's alias must be an identifier, got 'a b'"]));
  assert(same(P.importfrom("m.", "x y", 1n).validate(), ["an importfrom's module must be a dotted name, got 'm.'",
    "an importfrom's name must be an identifier, got 'x y'"]));
  assert(same(P.importfrom("m", "n", 1n, "").validate(), ["an importfrom needs an alias"])); // empty, not absent
  assert(same(P.let_("a b", 1n, 2n).validate(), ["a let's name must be an identifier, got 'a b'"]));
  assert(same(P.attribute(P.name("x"), "1").validate({ bound: ["x"] }), ["an attribute must be an identifier, got '1'"]));
  assert(P.importfrom("m", "n", P.name("n")).validate().length === 0
    && same(new (kindOf(P.DIALECT, "import"))(7n, null, P.constant(1n)).validate(), ["an import's module must be a str, got int"]));
  assert(same(P.binop("@", 1n, 2n).validate({ core: true }), ["'@' is not a core operation"]));
  assert(P.DIALECT.infer(P.binop("+", 1n, 2.5)) === PD.Float && P.DIALECT.infer(P.binop("+", true, 1n)) === PD.Int);
  assert(P.DIALECT.infer(P.binop("+", "a", "b")) === PD.Str && P.DIALECT.infer(P.unaryop("-", 2.0)) === PD.Float);
  assert(P.DIALECT.infer(P.binop("**", 2n, 3n)).name() === "int | float" && P.DIALECT.infer(P.unaryop("not", 1n)) === PD.Bool);
  assert(P.DIALECT.infer(P.compare("<", 1n, 2.5)) === PD.Bool && P.DIALECT.infer(P.attribute(P.name("x"), "a"), { x: D.Anything }) === D.Anything);
  raises(TypeError, () => P.DIALECT.infer(P.compare("<", "a", 1n)), "< cannot take (str, int)");
  assert(PD.of(new Uint8Array()) === PD.Bytes && PD.Numeric.includes(PD.Bool));
}

## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}, options: any = {}) =>
    PV.OfAny(expression, new PV.Scope(variables, options));
  const [a, t] = [P.name("a"), P.name("t")];
  assert(ev(P.boolop("or", 0n, "x")) === "x" && ev(P.boolop("and", "", 1n)) === "" && ev(P.boolop("and", 1n, 2n)) === 2n); // an operand, not a bool
  assert(ev(P.compare("==", 1n, 1.0)) === true && ev(P.binop("+", true, 1n)) === 2n);
  assert(ev(P.binop("-", P.binop("%", P.binop("//", 7n, 2n), 3n), P.binop("/", P.binop("**", 2n, 2n), 4n))) === -1.0);
  assert(ev(P.ifexp(P.compare(">", a, 1n), a, P.name("missing")), { a: 2n }) === 2n
    && ev(P.let_("a", 3n, P.binop("*", a, a))) === 9n);
  assert(ev(P.attribute(t, "age"), { t: ann }) === 30n && ev(P.call("hasattr", t, "email"), { t: bob }) === false); // properties are attributes
  assert(ev(P.call("getattr", t, "name"), { t: ann }) === "Ann" && ev(P.subscript(t, "x"), { t: { x: 1n } }) === 1n);
  assert(ev(P.call("hasattr", t, "_secret"), { t: ann }) === false);
  assert(ev(P.binop("+", P.binop("+", P.call("abs", P.unaryop("-", 2n)), P.call("len", "abc")), P.call("max", 1n, 2n))) === 7n
    && ev(P.call("f", 2n), { f: (v: bigint) => v + 1n }) === 3n);
  raises(AttributeError, () => ev(P.attribute(t, "email"), { t: bob }), "'Person' object has no attribute 'email'");
  raises(AttributeError, () => ev(P.attribute(t, "__class__"), { t: ann }), "attribute '__class__' is private");
  raises(NameError, () => ev(P.name("x")), "name 'x' is not defined");
  raises(NameError, () => ev(P.call("len", "a"), {}, { builtins: {} }), "name 'len' is not defined");
  function shout(value: string): string {
    return value.toUpperCase();
  }
  raises(TypeError, () => ev(P.call(P.subscript(t, "f"), "a"), { t: { f: shout } }),
    "calling 'shout' is not allowed by the scope"); // only what the scope provides, not what values hold
  raises(TypeError, () => ev(P.call(t), { t: 3n }), "calling 'int' is not allowed by the scope");

  // NumPy is Python's; a module of the scope stands in for it here.
  const masked = { masked: true };
  const numpy = { numpy: new PV.Module("numpy", {
    ma: new PV.Module("numpy.ma", { masked }), pi: Math.PI, _core: new PV.Module("numpy._core", {}) }) };
  assert(ev(P.import_("numpy.ma", P.attribute(P.attribute(P.name("numpy"), "ma"), "masked")), {}, { modules: numpy }) === masked); // binds numpy
  assert(ev(P.import_("numpy.ma", P.attribute(P.name("ma"), "masked"), "ma"), {}, { modules: numpy }) === masked
    && ev(P.import_("numpy", P.attribute(P.name("numpy"), "pi")), {}, { modules: numpy }) === Math.PI);
  raises(ImportError, () => ev(P.import_("numpy", 1n, "np")), "import of 'numpy' is not allowed by the scope"); // nothing is imported unless the scope allows it
  raises(ImportError, () => ev(P.import_("os", P.call("os.getcwd")), {}, { modules: numpy }), "import of 'os' is not allowed by the scope");
  raises(ImportError, () => ev(P.import_("numpy.nope", 1n), {}, { modules: numpy }), "No module named 'numpy.nope'");
  raises(ImportError, () => ev(P.import_("numpy._core", 1n), {}, { modules: numpy }), "No module named 'numpy._core'");
  raises(ImportError, () => ev(P.importfrom("numpy", "nope", 1n), {}, { modules: numpy }), "cannot import name 'nope' from 'numpy'");
  raises(ImportError, () => ev(P.importfrom("numpy", "_core", 1n), {}, { modules: numpy }), "cannot import name '_core' from 'numpy'");
  assert(PV.OfAny(P.binop("+", P.name("x"), 1n), { x: 1n }) === 2n); // an object is the variables of a scope

  // Python's rules, case by case. The Python suite runs this table against Python itself; here it runs against the
  // model of them.
  const n = P.name;
  const bytes = (...values: number[]) => new Uint8Array(values);
  const modules = { math: new PV.Module("math", { floor: (v: number) => BigInt(Math.floor(v)) }), ...numpy };
  const Squad = new S.OfObject.Builder().ref().properties((p) => p.name("ranks").of(
    (t) => t.as_indexed((i) => i.of(new S.OfNative.Data(BigInt))))).create();
  Proxies.register("Squad", Squad);
  const squad = (Proxies.Builders as any).Squad().ranks([1n, 2n, 3n]).create(); // a list property is an array of its values
  const VALUES: [F.Term, Record<string, unknown>, unknown][] = [ // (expression, variables, value): the value and its type
    [P.unaryop("not", n("v")), { v: null }, true], [P.unaryop("not", 0.0), {}, true], [P.unaryop("not", ""), {}, true],
    [P.unaryop("not", bytes()), {}, true], [P.unaryop("not", n("v")), { v: [] }, true], [P.unaryop("not", n("v")), { v: {} }, true],
    [P.unaryop("not", NaN), {}, false], [P.unaryop("not", n("v")), { v: { k: 1n } }, false],
    [P.compare("<", 1n, 1.5), {}, true], [P.compare(">", 2n, 1.5), {}, true], [P.compare("==", 1n, 1.5), {}, false],
    [P.compare("==", P.binop("+", P.binop("**", 2n, 60n), 1n), P.call("float", P.binop("**", 2n, 60n))), {}, false],
    [P.compare("<", 1n, Infinity), {}, true], [P.compare(">", 1n, -Infinity), {}, true], [P.compare("<", 1n, NaN), {}, false],
    [P.compare("<", 1.5, 2n), {}, true], [P.compare("==", NaN, NaN), {}, false], [P.compare("==", 3n, 3.0), {}, true],
    [P.compare("<", 3n, 3.5), {}, true], [P.compare("<", 2.5, 1.5), {}, false], [P.compare("<", true, 2n), {}, true],
    [P.compare("<", "a", "b"), {}, true], [P.compare("<", bytes(97), bytes(98)), {}, true], [P.compare("==", bytes(97), bytes(97)), {}, true],
    [P.compare("!=", "a", 1n), {}, true],
    [P.binop("+", "ab", "c"), {}, "abc"], [P.binop("+", bytes(97), bytes(98)), {}, bytes(97, 98)], [P.binop("*", "ab", 2n), {}, "abab"],
    [P.binop("*", 2n, bytes(120)), {}, bytes(120, 120)], [P.binop("*", "ab", -1n), {}, ""], [P.binop("//", -7n, 2n), {}, -4n],
    [P.binop("%", -7n, 2n), {}, 1n], [P.binop("%", 7n, -2n), {}, -1n], [P.binop("%", -7.5, 2n), {}, 0.5], [P.binop("%", 7.5, 2n), {}, 1.5],
    [P.binop("//", 7.5, 2n), {}, 3.0], [P.binop("**", 2n, -1n), {}, 0.5], [P.binop("**", 2.0, 3n), {}, 8.0],
    [P.binop("+", 1.5, 1n), {}, 2.5], [P.binop("*", 1.5, 2n), {}, 3.0], [P.binop("-", 3n, 1.5), {}, 1.5], [P.binop("/", 1n, 2n), {}, 0.5],
    [P.unaryop("-", true), {}, -1n], [P.unaryop("+", true), {}, 1n], [P.unaryop("-", 1.5), {}, -1.5], [P.unaryop("+", 1.5), {}, 1.5],
    [P.call("abs", true), {}, 1n], [P.call("abs", -1.5), {}, 1.5], [P.call("bool", ""), {}, false], [P.call("float", 1n), {}, 1.0],
    [P.call("float", " 1e3 "), {}, 1000.0], [P.call("float", "inf"), {}, Infinity], [P.call("float", "-inf"), {}, -Infinity],
    [P.call("str", P.call("float", "nan")), {}, "nan"], [P.call("int", " 42 "), {}, 42n], [P.call("int", 2.7), {}, 2n],
    [P.call("int", -2.7), {}, -2n], [P.call("int", true), {}, 1n], [P.call("len", bytes(97, 98)), {}, 2n],
    [P.call("len", n("v")), { v: [1n, 2n] }, 2n], [P.call("len", n("v")), { v: { a: 1n } }, 1n], [P.call("len", "héllo"), {}, 5n],
    [P.call("max", 1n, 2.5), {}, 2.5], [P.call("min", 3n, 1n, 2n), {}, 1n], [P.call("round", 2.5), {}, 2n],
    [P.call("round", 3.5), {}, 4n], [P.call("round", -0.5), {}, 0n], [P.call("round", 7n), {}, 7n], [P.call("str", 1.0), {}, "1.0"],
    [P.call("str", bytes(120)), {}, "b'x'"], [P.call("str", n("v")), { v: null }, "None"], [P.call("str", true), {}, "True"],
    [P.call("str", 7n), {}, "7"], [P.call("str", "s"), {}, "s"], [P.import_("math", P.call("math.floor", 2.5)), {}, 2n],
    [P.unaryop("-", false), {}, 0n], [P.unaryop("not", n("v")), { v: ann }, false], [P.compare("==", 1.5, 1.5), {}, true],
    [P.binop("/", 1.5, 2n), {}, 0.75], [P.binop("&", 12n, 10n), {}, 8n], [P.binop("|", 12n, true), {}, 13n],
    [P.binop("^", true, true), {}, false], [P.binop("&", true, false), {}, false], [P.binop("|", false, true), {}, true],
    [P.binop("<<", true, 3n), {}, 8n], [P.binop(">>", -7n, 1n), {}, -4n], [P.unaryop("~", 5n), {}, -6n],
    [P.binop("|", P.binop("**", 2n, 70n), 1n), {}, 2n ** 70n + 1n],
    [P.call("all", P.generator("p", n("v"), P.compare(">", n("p"), 0n))), { v: [1n, 2n] }, true],
    [P.call("all", P.generator("p", n("v"), n("p"))), { v: [] }, true], [P.call("any", n("v")), { v: [0n, 3n] }, true],
    [P.call("any", P.generator("p", n("v"), n("p"))), { v: [0n, ""] }, false],
    [P.call("sum", P.generator("p", n("v"), 1n, P.compare(">", n("p"), 0n))), { v: [1n, -1n, 2n] }, 2n],
    [P.call("sum", P.generator("c", "abc", 1n, P.compare("!=", n("c"), "b"), P.compare("!=", n("c"), "c"))), {}, 1n],
    [P.call("sum", n("v")), { v: Array(10).fill(0.1) }, 1.0], [P.call("sum", n("v")), { v: [1n, 0.1, 0.2, 0.3] }, 1.6],
    [P.call("sum", n("v")), { v: [1e+16, 1n, 1n] }, 1.0000000000000002e+16],
    [P.call("sum", n("v"), true), { v: Array(10).fill(0.1) }, 2.000000000000001], [P.call("sum", n("v")), { v: [true, true] }, 2n],
    [P.call("sum", n("v")), { v: [] }, 0n], [P.call("sum", n("v"), 1.5), { v: [] }, 1.5],
    [P.call("sum", n("v")), { v: [2n ** 62n, 2n ** 62n, 0.5] }, 9.223372036854776e+18],
    [P.call("sum", n("v")), { v: [(2n ** 63n) - 1n, 1n] }, 2n ** 63n],
    [P.call("sum", n("v")), { v: [-654.25, (-(2n ** 63n)) - 1n, -9n, (2n ** 63n) + 1n] }, -663.25],
    [P.call("sum", n("v"), 2n ** 64n), { v: [1n, 0.5] }, 1.8446744073709552e+19], [P.call("min", n("v")), { v: [3n, 1n, 2n] }, 1n],
    [P.call("max", P.generator("p", n("v"), P.binop("*", n("p"), 2n))), { v: [1n, 2.5] }, 5.0], [P.call("max", "abc"), {}, "c"],
    [P.call("min", P.call("set", n("v"))), { v: [2n, 1n] }, 1n], [P.call("len", P.call("set", n("v"))), { v: [1n, 1.0, true, 2n] }, 2n],
    [P.call("len", P.call("set")), {}, 0n], [P.unaryop("not", P.call("set")), {}, true],
    [P.compare("in", 2n, n("v")), { v: [1n, 2n] }, true], [P.compare("in", "b", "abc"), {}, true],
    [P.compare("in", "k", n("v")), { v: new Map([["k", 1n]]) }, true], [P.compare("not in", 3n, n("v")), { v: [1n, 2n] }, true],
    [P.compare("in", 1.0, n("v")), { v: [1n] }, true], [P.compare("in", 98n, new Uint8Array([97, 98])), {}, true],
    [P.compare("in", new Uint8Array([98]), new Uint8Array([97, 98])), {}, true],
    [P.compare("in", new Uint8Array([98, 97]), new Uint8Array([97, 98])), {}, false],
    [P.compare("in", n("w"), n("v")), { v: [[1n]], w: [1n] }, true], [P.compare("in", 1n, P.call("set", n("v"))), { v: [1n] }, true],
    [P.compare("in", 3n, P.generator("p", n("v"), n("p"))), { v: [1n, 2n] }, false], [P.index(n("v"), 1n), { v: [10n, 20n] }, 20n],
    [P.index(n("v"), -1n), { v: [10n, 20n] }, 20n], [P.index("abc", 1n), {}, "b"], [P.index(new Uint8Array([97, 98]), 0n), {}, 97n],
    [P.index(n("v"), true), { v: [10n, 20n] }, 20n], [P.index(n("v"), 1.0), { v: new Map([[1n, "x"]]) }, "x"],
    [P.index(n("v"), "k"), { v: new Map([["k", 1n]]) }, 1n], [P.call("all", P.index(n("v"), 0n)), { v: [[1n, 1n]] }, true],
    [P.call("sum", bytes(1, 2)), {}, 3n], [P.call("max", bytes(97, 98)), {}, 98n],
    [P.call("len", P.call("set", n("v"))), { v: { a: 1n, b: 2n } }, 2n], [P.index(n("v"), "b"), { v: { a: 1n, b: 2n } }, 2n],
    [P.call("sum", P.attribute(n("v"), "ranks")), { v: squad }, 6n], [P.index(P.attribute(n("v"), "ranks"), -1n), { v: squad }, 3n],
  ];
  const pyEqual = (a: unknown, b: unknown) => Repr.typeName(a) === Repr.typeName(b)
    && (a instanceof Uint8Array ? F.sameNative(a, b) : a === b);
  for (const [expression, variables, expected] of VALUES) {
    const value = PV.OfAny(expression, new PV.Scope(variables, { modules }));
    assert(pyEqual(value, expected), `${PText.ToText(expression)}: ${Repr.repr(value)}, not ${Repr.repr(expected)}`);
  }
  const ERRORS: [F.Term, Record<string, unknown>, Function, string][] = [ // (expression, variables, error, message)
    [P.compare("<", "a", 1n), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"],
    [P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"],
    [P.binop("-", "a", 1n), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"],
    [P.binop("&", 1.5, 1n), {}, TypeError, "unsupported operand type(s) for &: 'float' and 'int'"],
    [P.binop("<<", 1n, -1n), {}, ValueError, "negative shift count"], [P.binop(">>", 1n, -1n), {}, ValueError, "negative shift count"],
    [P.unaryop("~", 1.5), {}, TypeError, "bad operand type for unary ~: 'float'"],
    [P.binop("/", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("//", 1.5, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("%", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("**", 0.0, -1n), {}, ZeroDivisionError, "zero to a negative power"],
    [P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"],
    [P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"],
    [P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"],
    [P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"],
    [P.call("int", NaN), {}, ValueError, "cannot convert float NaN to integer"],
    [P.call("int", Infinity), {}, OverflowError, "cannot convert float infinity to integer"],
    [P.call("len", 1n), {}, TypeError, "object of type 'int' has no len()"],
    [P.call("max", 1n, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"],
    [P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"],
    [P.call("float", n("v")), { v: null }, TypeError, "float() argument must be a string or a real number, not 'NoneType'"],
    [P.call("int", n("v")), { v: null }, TypeError,
      "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"],
    [P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"],
    [P.subscript(n("t"), "y"), { t: { x: 1n } }, KeyError, "'y'"],
    [P.subscript(n("t"), "x"), { t: 3n }, TypeError, "'int' object is not subscriptable"],
    [P.call(P.subscript(n("t"), "f")), { t: { f: [() => 1n][0] } }, TypeError, "calling '<lambda>' is not allowed by the scope"],
    [P.call("len", P.generator("p", n("v"), n("p"))), { v: [1n] }, TypeError, "object of type 'generator' has no len()"],
    [P.call("sum", n("v")), { v: ["a"] }, TypeError, "unsupported operand type(s) for +: 'int' and 'str'"],
    [P.call("sum", n("v")), { v: [0.5, "a"] }, TypeError, "unsupported operand type(s) for +: 'float' and 'str'"],
    [P.call("sum", n("v"), "a"), { v: [1n] }, TypeError, "sum() can't sum strings [use ''.join(seq) instead]"],
    [P.call("sum", n("v"), new Uint8Array([])), { v: [1n] }, TypeError, "sum() can't sum bytes [use b''.join(seq) instead]"],
    [P.call("sum", n("v")), { v: [0.5, 10n ** 400n] }, OverflowError, "int too large to convert to float"],
    [P.call("min", n("v")), { v: [] }, ValueError, "min() iterable argument is empty"],
    [P.call("max", P.generator("p", n("v"), n("p"))), { v: [] }, ValueError, "max() iterable argument is empty"],
    [P.call("max"), {}, TypeError, "max expected at least 1 argument, got 0"],
    [P.call("min", 1n), {}, TypeError, "'int' object is not iterable"],
    [P.call("set", n("v")), { v: [[1n]] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.call("all", 5n), {}, TypeError, "'int' object is not iterable"],
    [P.generator("p", 5n, n("p")), {}, TypeError, "'int' object is not iterable"],
    [P.compare("in", 1n, 5n), {}, TypeError, "argument of type 'int' is not a container or iterable"],
    [P.compare("in", 1n, "abc"), {}, TypeError, "'in <string>' requires string as left operand, not int"],
    [P.compare("in", "a", new Uint8Array([97, 98])), {}, TypeError, "a bytes-like object is required, not 'str'"],
    [P.compare("in", 256n, new Uint8Array([97, 98])), {}, ValueError, "byte must be in range(0, 256)"],
    [P.compare("in", n("w"), n("v")), { v: new Map([[1n, 2n]]), w: [1n] }, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"],
    [P.compare("in", n("w"), P.call("set", n("v"))), { v: [1n], w: [1n] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.index(n("v"), 5n), { v: [1n] }, IndexError, "list index out of range"],
    [P.index(n("v"), -3n), { v: [1n, 2n] }, IndexError, "list index out of range"],
    [P.index("abc", 5n), {}, IndexError, "string index out of range"],
    [P.index(new Uint8Array([97]), 3n), {}, IndexError, "index out of range"],
    [P.index(n("v"), P.binop("**", 2n, 70n)), { v: [1n] }, IndexError, "cannot fit 'int' into an index-sized integer"],
    [P.index(n("v"), 1.5), { v: [1n] }, TypeError, "list indices must be integers or slices, not float"],
    [P.index("abc", 1.5), {}, TypeError, "string indices must be integers, not 'float'"],
    [P.index(new Uint8Array([97, 98]), 1.5), {}, TypeError, "byte indices must be integers or slices, not float"],
    [P.index(5n, 0n), {}, TypeError, "'int' object is not subscriptable"],
    [P.index(n("v"), 5n), { v: new Map([["k", 1n]]) }, KeyError, "5"],
    [P.index(n("v"), n("w")), { v: new Map([["k", 1n]]), w: ["k"] }, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"],
  ];
  for (const [expression, variables, error, message] of ERRORS) {
    raises(error, () => PV.OfAny(expression, new PV.Scope(variables, { modules })), message);
  }
  assert(PV.OfAny(P.importfrom("numpy.ma", "masked", n("masked")), new PV.Scope({}, { modules })) === masked);
  // A mapping is a Map or a plain object.
  assert(ev(P.subscript(t, "x"), { t: new Map([["x", 1n]]) }) === 1n && ev(P.unaryop("not", t), { t: new Map() }) === true);
  raises(KeyError, () => ev(P.subscript(t, "y"), { t: new Map() }), "'y'");
  assert(ev(P.call("len", t), { t: new Map([["x", 1n]]) }) === 1n);
}

## DIA-03 · Matlab: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
const s = M.identifier("s");
const matlabRule = M.binary("&&", M.binary(">=", M.field(s, "age"), 18n), M.call("isfield", s, "email"));
{
  assert(MText.ToText(matlabRule) === 's.age >= 18 && isfield(s, "email")');
  assert(MText.ToText(M.binary("-", 1n, M.binary("-", 2n, 3n))) === "1 - (2 - 3)"
    && MText.ToText(M.binary("-", M.binary("-", 1n, 2n), 3n)) === "1 - 2 - 3");
  assert(MText.ToText(M.binary(".*", M.binary("+", 1n, 2n), M.unary("-", M.unary("~", s)))) === "(1 + 2) .* -~s");
  assert(MText.ToText(M.unary("-", M.binary("+", 1n, 2n))) === "-(1 + 2)" && MText.ToText(M.unary("~", -1n)) === "~-1");
  assert(MText.ToText(M.binary("||", M.binary("&&", true, false), M.binary("&&", 1n, 0n))) === "true && false || 1 && 0");
  assert(same([NaN, Infinity, -Infinity, -2.5, 'say "hi"', 7n].map((v) => MText.ToText(M.constant(v))), [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]));
  assert(MText.ToText(M.field(M.field(s, "a"), "b")) === "s.a.b");
  assert(M.DIALECT.infer(matlabRule, { s: MD.Struct }) === MD.Logical);
  assert(M.DIALECT.infer(M.binary("+", "a", 1n)) === MD.String && M.DIALECT.infer(M.binary("+", true, 1n)) === MD.Double);
  assert(MD.Double.contains(1n) && MD.Struct.contains({}) && !MD.Struct.contains("s") && MD.of(1n) === MD.Double);
  assert(same(M.binary("^", 1n, 2n).validate({ core: true }), ["'^' is not a core operation"]));
}

## DIA-04 · Matlab evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
{
  assert(MV.OfAny(matlabRule, { s: ann }) === true && MV.OfAny(matlabRule, { s: bob }) === false);
  assert(MV.OfAny(matlabRule, { s: { age: 20n, email: "x" } }) === true); // a mapping is a struct
  assert(MV.OfAny(M.field(s, "age"), { s: ann }) === 30 && typeof MV.OfAny(18n) === "number"); // numbers are doubles
  assert(MV.OfAny(M.binary("==", 1n, true)) === true && MV.OfAny(M.binary("+", true, 1n)) === 2);
  assert(MV.OfAny(M.binary("<", "apple", "banana")) === true && MV.OfAny(M.binary("==", "5", 5n)) === true);
  assert(MV.OfAny(M.binary("==", "true", true)) === true && MV.OfAny(M.binary("~=", "1.5", 1.5)) === false);
  assert(MV.OfAny(M.binary("+", "a", 1n)) === "a1" && MV.OfAny(M.binary("+", "x", 0.125)) === "x0.125");
  assert(same([NaN, Infinity, -Infinity, 12345.6, 1.5e-7, 123456.7, true, false].map((v) => MV.OfAny(M.binary("+", "x", v))), [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"])); // numbers as MATLAB writes them
  assert(MV.OfAny(matlabRule, { s: new Map<string, unknown>([["age", 20n], ["email", "x"]]) }) === true); // a Map is a struct too
  assert(same(["~=", "<=", ">", ">=", "-", ".*"].map((op) => MV.OfAny(M.binary(op, 2n, 3n))), [true, true, false, false, -1, 6]));
  assert(MV.OfAny(M.binary("&&", false, M.identifier("missing"))) === false); // short-circuits
  assert(MV.OfAny(M.binary("||", 2n, M.identifier("missing"))) === true && MV.OfAny(M.binary("||", 0n, 0n)) === false);
  assert(MV.OfAny(M.unary("~", 0n)) === true && MV.OfAny(M.unary("-", true)) === -1);
  assert(MV.OfAny(M.call("isfield", 3n, "x")) === false && MV.OfAny(M.call("isfield", s, 1n), { s: ann }) === false);
  raises(KeyError, () => MV.OfAny(M.field(s, "email"), { s: bob }), 'Unrecognized field name "email".');
  raises(TypeError, () => MV.OfAny(M.field(s, "age"), { s: 3n }), "Dot indexing is not supported for variables of this type.");
  raises(NameError, () => MV.OfAny(M.identifier("missing")), "Unrecognized function or variable 'missing'.");
  raises(TypeError, () => MV.OfAny(M.binary("&&", "yes", true)), "Operands to the && operator must be convertible to logical scalar values.");
  raises(ValueError, () => MV.OfAny(M.unary("~", NaN)), "NaN's cannot be converted to logicals.");
  raises(TypeError, () => MV.OfAny(M.binary("-", "a", 1n)), "Operator '-' is not supported for operands of type 'string' and 'double'.");
  raises(TypeError, () => MV.OfAny(M.binary(".*", s, true), { s: ann }), "Operator '.*' is not supported for operands of type 'struct' and 'logical'.");
  raises(TypeError, () => MV.OfAny(M.unary("-", "a")), "Operator '-' is not supported for operands of type 'string'.");
}

## DIA-05 · Matlab resolves functions from imports, the path and packages that the scope provides

In [ ]:
{
  const geo = { dist: (a: number, b: number) => (a * a + b * b) ** 0.5, area: (r: number) => 3 * r * r };
  const scope = new MV.Scope({ r: 2n }, { functions: { twice: (v: number) => 2 * v },
    packages: { geo, "geo.more": { area: () => 0n } } });
  const wild = M.import_("geo.*", M.binary("+", M.call("dist", 3n, 4n), M.call("area", M.identifier("r"))));
  assert(MText.ToText(wild) === "import geo.*\ndist(3, 4) + area(r)" && wild.validate({ bound: ["r"] }).length === 0);
  assert(same(wild.validate({ bound: ["r"], core: true }), ["body: left: 'dist' is not a core operation",
    "body: right: 'area' is not a core operation"]));
  assert(MV.OfAny(wild, scope) === 17);
  assert(MV.OfAny(M.import_("geo.dist", M.call("dist", 6n, 8n)), scope) === 10);
  assert(MV.OfAny(M.call("geo.more.area", 1n), scope) === 0 && MV.OfAny(M.call("twice", 4n), scope) === 8); // an int result is a double
  assert(MV.OfAny(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5n))), scope) === 0); // innermost wins
  raises(NameError, () => MV.OfAny(M.call("dist", 3n, 4n), scope), "Unrecognized function or variable 'dist'."); // not imported
  raises(ImportError, () => MV.OfAny(M.import_("os.system", 1n), scope), "Import argument 'os.system' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(M.import_("geo.nope", 1n), scope), "Import argument 'geo.nope' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(M.import_("nope.*", 1n), scope), "Import argument 'nope.*' cannot be found or cannot be imported.");
  raises(ValueError, () => MText.ToText(M.binary("+", 1n, M.import_("geo.*", 2n))), "an import can only enclose the whole expression");
  assert(same(M.import_("geo", 1n).validate(), ["an import's name must be pkg.name or pkg.*, got 'geo'"]));
  assert(same(M.import_("geo.*.x", 1n).validate(), ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]));
  assert(same(new (kindOf(M.DIALECT, "import"))(7n, M.constant(1n)).validate(), ["an import's name must be a str, got int"]));
}

## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
const r = X.name("r");
const excelRule = X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18n),
  X.function("NOT", X.function("ISERROR", X.field(r, "email")))));
{
  assert(XText.ToText(excelRule) === "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))");
  assert(XText.ToText(X.infix("*", X.infix("+", 1n, 2n), X.prefix("-", X.infix("-", 3n, 4n)))) === "=(1 + 2) * -(3 - 4)");
  assert(XText.ToText(X.infix("=", X.infix("-", 1n, X.infix("-", 2n, 3n)), -1n)) === "=1 - (2 - 3) = -1");
  assert(XText.ToText(X.prefix("-", X.field(r, "first name"))) === "=-r.[first name]");
  assert(same([true, false, 'say "hi"', Infinity, 2.5, 3n].map((v) => XText.ToText(X.constant(v))), [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]));
  assert(excelRule.validate({ bound: ["r"], core: true }).length === 0
    && same(X.function("VLOOKUP", 1n, 2n).validate({ core: true }), ["'VLOOKUP' is not a core operation"]));
  assert(X.DIALECT.infer(excelRule, { r: XD.Record }) === XD.Logical && X.DIALECT.infer(X.infix("+", "1", true)) === XD.Number);
  assert(XD.of(1n) === XD.Number && XD.Errors.contains(XD.Error.of("#N/A")) && String(XD.Error.of("#N/A")) === "#N/A");
  assert(XD.Record.contains(ann) && !XD.Record.contains(1n));
}

## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
{
  const Error = XV.Error;
  assert(XV.OfAny(excelRule, { r: ann }) === true && XV.OfAny(excelRule, { r: bob }) === false); // ISERROR finds the #FIELD!
  assert(XV.OfAny(X.field(r, "email"), { r: bob }) === Error.of("#FIELD!") && XV.OfAny(X.name("nope")) === Error.of("#NAME?"));
  assert(XV.OfAny(X.field(r, "age"), { r: { age: 5n } }) === 5 && XV.OfAny(X.field(r, "x"), { r: 3n }) === Error.of("#VALUE!"));
  assert(XV.OfAny(X.field(X.field(r, "a"), "b"), { r: {} }) === Error.of("#FIELD!")); // errors propagate
  assert(XV.OfAny(X.infix("+", "1", true)) === 2 && XV.OfAny(X.infix("*", "x", 1n)) === Error.of("#VALUE!")); // coercion
  assert(XV.OfAny(X.infix("-", 1n, X.infix("+", X.name("u"), 1n))) === Error.of("#NAME?"));
  assert(XV.OfAny(X.prefix("-", "2")) === -2 && XV.OfAny(X.prefix("-", X.name("u"))) === Error.of("#NAME?"));
  assert(XV.OfAny(X.infix(">", "a", 1n)) === true && XV.OfAny(X.infix(">", true, "z")) === true); // number < text < logical
  assert(XV.OfAny(X.infix("=", "a", "A")) === true && XV.OfAny(X.infix("<>", 1n, "1")) === true); // case; no coercion
  assert(same(["<", "<=", ">=", "-"].map((op) => XV.OfAny(X.infix(op, 2n, 3n))), [true, true, false, -1]));
  assert(XV.OfAny(X.infix("=", 2n, 2n)) === true && XV.OfAny(X.infix("+", false, 1n)) === 1);
  assert(XV.OfAny(X.infix("+", "inf", 1n)) === Error.of("#VALUE!") && XV.OfAny(X.infix("+", " 1e3 ", 1n)) === 1001); // decimal text
  assert(XV.OfAny(X.infix("+", r, 1n), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(X.field(r, "age"), { r: new Map([["age", 5n]]) }) === 5); // a Map is a record too
  assert(XV.OfAny(X.infix("=", X.name("u"), 1n)) === Error.of("#NAME?") && XV.OfAny(X.infix("=", r, 1n), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(X.function("AND", 1n, true)) === true && XV.OfAny(X.function("OR", 0n, false)) === false);
  assert(XV.OfAny(X.function("AND", false, X.name("u"))) === Error.of("#NAME?")); // AND evaluates every argument
  assert(XV.OfAny(X.function("OR", "yes", true)) === Error.of("#VALUE!") && XV.OfAny(X.function("NOT", 0n)) === true);
  assert(XV.OfAny(X.function("NOT", X.name("u"))) === Error.of("#NAME?"));
  assert(XV.OfAny(X.function("IF", true, 1n, X.name("u"))) === 1 && XV.OfAny(X.function("IF", 0n, 1n, "no")) === "no");
  assert(XV.OfAny(X.function("IF", X.name("u"), 1n, 2n)) === Error.of("#NAME?")); // IF evaluates only the branch it takes
  assert(XV.OfAny(X.function("ISERROR", 1n)) === false && XV.OfAny(X.let_("x", 2n, X.infix("*", X.name("x"), 3n))) === 6);
}

## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
{
  const Error = XV.Error;
  const rates = new XV.Workbook({}, { Rates: { B2: 0.5 } }, { name: "Rates.xlsx" });
  const book = new XV.Workbook({ base: 10n }, { Sheet1: { a1: 2n, $A$2: 3n }, "My Sheet": { C3: "x" } },
    { books: { "Rates.xlsx": rates }, add_ins: { double: (v: number) => v * 2 } });
  const total = X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base")));
  assert(XText.ToText(total) === "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" && XV.OfAny(total, book) === 21);
  assert(total.validate({ bound: ["base"] }).length === 0
    && same(total.validate({ bound: ["base"], core: true }), ["right: 'DOUBLE' is not a core operation"]));
  assert(XText.ToText(X.cell("C3", "My Sheet")) === "='My Sheet'!C3" && XV.OfAny(X.cell("C3", "My Sheet"), book) === "x");
  assert(XText.ToText(X.cell("A1", "Bob's", "My Book.xlsx")) === "='[My Book.xlsx]Bob''s'!A1");
  assert(XV.OfAny(X.cell("$a$2"), book) === 3 && XV.OfAny(X.cell("Z9"), book) === 0); // an empty cell is 0
  assert(XV.OfAny(X.cell("A1", "Sheet1", "Book1"), book) === 2); // this book, named
  assert(XV.OfAny(X.cell("A1", "Nope"), book) === Error.of("#REF!") && XV.OfAny(X.cell("A1", "S", "Other.xlsx"), book) === Error.of("#REF!"));
  assert(XV.OfAny(X.function("NOPE", 1n), book) === Error.of("#NAME?")
    && XV.OfAny(X.function("double", X.name("u")), book) === Error.of("#NAME?")); // add-ins ignore case; errors in their arguments propagate
  assert(XV.OfAny(X.cell("A1"), new XV.Workbook()) === Error.of("#REF!")); // a workbook has no sheet until given one
  assert(XV.OfAny(X.name("x"), { x: 1n }) === 1 && new XV.Workbook().sheet === "Sheet1");
  raises(ValueError, () => new XV.Workbook({}, { Sheet1: { total: 1n } }), "not a cell address: 'total'");
  assert(same(X.cell("A0").validate(), ["a cell's address must be a column and a row, like A1, got 'A0'"]));
  assert(same(X.cell("A1", null, "B.xlsx").validate(), ["a cell in another book needs a sheet"]));
  assert(X.DIALECT.infer(X.cell("A1")) === D.Anything && X.address("$b$12") === "B12" && X.address("B") === null);
}

## DIA-09 · Latex: notation in math mode, rendered and checked, with no evaluator

In [ ]:
{
  const [L, LD] = [Latex.Expressions, Latex.Domains];
  const [a, self] = [L.symbol("a"), L.symbol("this")];
  const notation = L.where("a", L.member(self, "age"), L.binary("\\land", L.binary("\\geq", a, 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", self, "email")), L.binary(">", L.frac(a, 2n), 1.5))));
  assert(LText.ToText(notation) === "a \\geq 18 \\land (\\lnot \\operatorname{has}(\\mathit{this}, \\text{email}) \\lor \\frac{a}{2} > 1.5) "
    + "\\quad \\text{where } a = \\mathit{this}.\\mathit{age}");
  assert(LText.ToText(L.binary("\\cdot", L.binary("+", 1n, 2n), L.unary("-", L.binary("-", 3n, -4n)))) === "(1 + 2) \\cdot -(3 - -4)");
  assert(LText.ToText(L.binary("\\implies", L.binary("\\implies", 1n, 2n), L.binary("\\implies", 3n, 4n)))
    === "(1 \\implies 2) \\implies 3 \\implies 4"); // right-associative
  assert(LText.ToText(L.binary("<", L.binary("<", 1n, 2n), 3n)) === "(1 < 2) < 3"); // comparisons do not chain
  assert(LText.ToText(L.member(L.binary("+", self, 1n), "x_y")) === "(\\mathit{this} + 1).\\mathit{x\\_y}");
  assert(LText.ToText(L.where("b", 1n, L.where("a", 2n, a))) === "(a \\quad \\text{where } a = 2) \\quad \\text{where } b = 1");
  assert(same([NaN, Infinity, -Infinity, 1e-7, 1e16, -2n, true, false, "a_b {c}"].map((v) => LText.ToText(L.constant(v))), [
    "\\mathrm{NaN}", "\\infty", "-\\infty", "1 \\times 10^{-7}", "1 \\times 10^{16}", "-2", "\\mathrm{true}",
    "\\mathrm{false}", "\\text{a\\_b \\{c\\}}"]));
  assert(LText.ToText(L.constant("\\$&#%~^")) === "\\text{\\textbackslash{}\\$\\&\\#\\%\\textasciitilde{}\\textasciicircum{}}");
  assert(LText.ToText(L.binary("\\cdot", L.constant(1e16), 2n)) === "1 \\times 10^{16} \\cdot 2");
  assert(notation.validate({ bound: ["this"], core: true }).length === 0
    && same(L.binary("\\times", 1n, 2n).validate({ core: true }), ["'\\\\times' is not a core operation"]));
  assert(L.function("max", 1n, 2n).validate().length === 0 && same(L.function("has", 1n).validate(), ["has takes 2 arguments, got 1"]));
  assert(L.DIALECT.infer(notation, { this: D.Anything }) === LD.Truth && L.DIALECT.infer(L.frac(1n, 2.5)) === LD.Number);
  raises(TypeError, () => L.DIALECT.infer(L.binary("\\land", 1n, true)), "\\land cannot take (number, truth)");
  assert(LD.of("x") === LD.Text && !("Evaluators" in Latex)); // notation is not evaluated
}

## DIA-10 · Bitwise operations: Python's operators, and MATLAB's and Excel's bit functions

In [ ]:
{
  // Python: rendered with Python's precedence, and evaluated by its rules (DIA-02's tables have more).
  const [a, b, c, d, x] = ["a", "b", "c", "d", "x"].map((name) => P.name(name)) as [any, any, any, any, any];
  const sources: [unknown, string][] = [
    [P.binop("|", a, P.binop("^", b, P.binop("&", c, P.binop(">>", P.binop("<<", d, 1n), 2n)))), "a | b ^ c & d << 1 >> 2"],
    [P.binop("&", P.binop("|", a, b), c), "(a | b) & c"], [P.binop("+", P.unaryop("~", x), 1n), "~x + 1"],
    [P.binop("+", P.binop("<<", a, b), c), "(a << b) + c"], [P.compare("==", P.binop("&", x, 1n), 1n), "x & 1 == 1"],
    [P.unaryop("-", P.unaryop("~", x)), "-~x"],
  ];
  for (const [expression, source] of sources) assert(PText.ToText(expression) === source, source);
  const mixed = P.binop("^", P.binop("|", P.binop("&", x, 12n), P.binop(">>", x, 2n)), P.binop("<<", x, 1n));
  assert(PText.ToText(mixed) === "(x & 12 | x >> 2) ^ x << 1" && PV.OfAny(mixed, { x: 13n }) === 21n);
  assert(P.DIALECT.infer(P.binop("&", true, false)) === PD.Bool && P.DIALECT.infer(P.binop("<<", 1n, true)) === PD.Int);
  assert(P.DIALECT.infer(P.unaryop("~", x), { x: PD.Bool }) === PD.Int);
  // MATLAB: the bit functions take doubles holding integers of 53 bits.
  assert(MV.OfAny(M.call("bitand", 12n, 10n)) === 8 && MV.OfAny(M.call("bitor", 12n, true)) === 13);
  assert(MV.OfAny(M.call("bitxor", 12n, 10n)) === 6 && MV.OfAny(M.call("bitshift", 1n, 3n)) === 8);
  assert(MV.OfAny(M.call("bitshift", 16n, M.unary("-", 2n))) === 4 && MV.OfAny(M.call("bitshift", 2 ** 52, 1n)) === 0); // beyond 53 bits
  assert(MText.ToText(M.call("bitshift", M.identifier("x"), M.unary("-", 2n))) === "bitshift(x, -2)");
  assert(M.DIALECT.infer(M.call("bitand", 1n, true)) === MD.Double);
  for (const call of [M.call("bitand", -1n, 1n), M.call("bitand", 1.5, 1n), M.call("bitor", 2 ** 54, 1n), M.call("bitshift", 1n, 0.5)]) {
    raises(ValueError, () => MV.OfAny(call), "Double inputs must have integer values in the range of ASSUMEDTYPE.");
  }
  raises(TypeError, () => MV.OfAny(M.call("bitor", "a", 1n)), "Undefined function 'bitor' for input arguments of type 'string'.");
  // Excel: the bit functions take integers below 2^48, converted as arithmetic converts; otherwise #NUM!.
  assert(XV.OfAny(X.function("BITAND", 12n, 10n)) === 8 && XV.OfAny(X.function("BITOR", "12", true)) === 13);
  assert(XV.OfAny(X.function("BITXOR", 12n, 10n)) === 6 && XV.OfAny(X.function("BITLSHIFT", 1n, 3n)) === 8);
  assert(XV.OfAny(X.function("BITRSHIFT", 16n, 2n)) === 4 && XV.OfAny(X.function("BITRSHIFT", 1n, -3n)) === 8);
  assert(XV.OfAny(X.function("BITLSHIFT", 16n, -2n)) === 4 && XText.ToText(X.function("BITXOR", X.name("x"), 1n)) === "=BITXOR(x, 1)");
  for (const formula of [X.function("BITAND", -1n, 1n), X.function("BITAND", 1.5, 1n), X.function("BITOR", 2n ** 48n, 1n),
    X.function("BITLSHIFT", 1n, 54n), X.function("BITLSHIFT", 1n, 48n), X.function("BITRSHIFT", 1n, 0.5)]) {
    assert(XV.OfAny(formula) === XV.Error.of("#NUM!"), XText.ToText(formula));
  }
  assert(XV.OfAny(X.function("BITAND", "a", 1n)) === XV.Error.of("#VALUE!"));
  assert(XV.OfAny(X.function("BITXOR", 1n, X.name("nope"))) === XV.Error.of("#NAME?"));
  assert(X.DIALECT.infer(X.function("BITAND", 1n, 2n)) === XD.Number);
}

## DIA-11 · Ccpp: C and C++ expressions over their arithmetic types, with C's precedence, typed constants and casts

In [ ]:
{
  const [x, y, me] = [C.identifier("x"), C.identifier("y"), C.identifier("this")];
  const cases: [unknown, string][] = [
    [C.binary("+", x, C.binary("*", y, 2n)), "x + y * 2"], [C.binary("*", C.binary("+", x, y), 2n), "(x + y) * 2"],
    [C.binary("-", x, C.binary("-", y, 1n)), "x - (y - 1)"], [C.binary("<<", C.binary("+", x, 1n), 2n), "x + 1 << 2"],
    [C.binary("&", C.binary("==", x, 1n), y), "x == 1 & y"], [C.binary("==", C.binary("&", x, 1n), y), "(x & 1) == y"],
    [C.binary("||", C.binary("&&", x, y), C.unary("!", x)), "x && y || !x"],
    [C.binary("|", C.binary("^", x, y), C.binary("%", x, y)), "x ^ y | x % y"],
    [C.binary("<", C.binary(">>", x, 1n), C.binary("/", y, 2n)), "x >> 1 < y / 2"],
    [C.conditional(x, y, C.conditional(y, 1n, 2n)), "x ? y : y ? 1 : 2"], [C.conditional(C.conditional(x, y, 1n), 2n, 3n), "(x ? y : 1) ? 2 : 3"],
    [C.unary("-", C.unary("-", x)), "- -x"], [C.unary("-", -1n), "- -1"], [C.unary("~", C.unary("!", x)), "~!x"],
    [C.cast("uint8_t", C.binary("+", x, 1n)), "(uint8_t)(x + 1)"], [C.unary("-", C.cast("int", x)), "-(int)x"],
    [C.member(C.member(me, "a"), "b", "->"), "this.a->b"], [C.subscript(C.member(me, "ports"), C.binary("+", x, 1n)), "this.ports[x + 1]"],
    [C.call("f", x, C.binary("+", y, 1n)), "f(x, y + 1)"], [C.member(C.binary("+", x, y), "a"), "(x + y).a"],
    [C.constant(5n, "unsigned int"), "5u"], [C.constant(5n, "unsigned long"), "5ul"], [C.constant(5n, "long long"), "5ll"],
    [C.constant(5n, "unsigned long long"), "5ull"], [C.constant(5n, "long"), "5l"], [C.constant(5n, "int"), "5"],
    [C.constant(1.5, "float"), "1.5f"], [C.constant("1.5", "long double"), "1.5L"], [C.constant("1", "long double"), "1.0L"],
    [C.constant("Infinity", "long double"), "INFINITY"], [C.constant(5n, "uint8_t"), "(uint8_t)5"], [C.constant(2.5, "double"), "2.5"],
    [C.constant(1.0), "1.0"], [C.constant(1e300), "1e+300"], [C.constant(NaN), "NAN"], [C.constant(-Infinity), "-INFINITY"], [C.constant(Infinity), "INFINITY"],
    [C.constant(true), "true"], [C.constant(false), "false"], [C.constant('a"\\\n\t\r'), '"a\\"\\\\\\n\\t\\r"'],
    [C.constant(-5n), "-5"], [C.binary("-", x, -5n), "x - -5"],
  ];
  for (const [expression, source] of cases) assert(CText.ToText(expression) === source, `${source}: ${CText.ToText(expression)}`);
  // Validation: types are C's arithmetic types, and hold the constant's value.
  assert(same(C.constant(5n, "uint9_t").validate(), ["a constant's type must be one of C's arithmetic types, got 'uint9_t'"]));
  assert(same(C.constant(300n, "uint8_t").validate(), ["a constant of uint8_t cannot hold 300"]));
  assert(same(C.cast("word", x).validate({ bound: ["x"] }), ["a cast's type must be one of C's arithmetic types, got 'word'"]));
  assert(same(C.member(x, "a", "=>").validate({ bound: ["x"] }), ["a member's operator must be '.' or '->', got '=>'"]));
  assert(same(C.binary("**", x, 1n).validate({ bound: ["x"], core: true }), ["'**' is not a core operation"]));
  assert(C.call("f", x).validate({ bound: ["x"], core: true }).length === 0); // functions are the scope's
  assert(same(new (C.DIALECT.kinds().get("cast") as any)().validate(), ["a cast needs a type", "a cast needs an operand"]));
  // Types under LP64, the integer promotions and the usual arithmetic conversions.
  const T = (name: string) => CD.TYPES.get(name) as Ccpp.Domains.CType;
  assert(T("int32_t").equals(T("int")) && T("int32_t").name() === "int32_t" && String(T("size_t")) === "size_t" && T("size_t").equals(T("unsigned long")));
  assert(CD.promoted(T("uint8_t")) === CD.Int && CD.promoted(CD.Bool) === CD.Int && CD.promoted(T("long")) === T("long"));
  assert(CD.common(CD.Int, T("unsigned int")).name() === "unsigned int" && CD.common(T("long"), T("unsigned int")).name() === "long");
  assert(CD.common(T("float"), T("long")).name() === "float" && CD.common(CD.Double, T("long double")).name() === "long double");
  assert(CD.common(T("unsigned long"), T("long long")).equals(T("unsigned long")) && CD.common(T("short"), T("char")) === CD.Int);
  assert(CD.common(CD.Int, T("long long")).name() === "long long" && CD.common(T("long"), CD.Int).name() === "long");
  const ofs = [1n, 2n ** 40n, 2n ** 63n, true, 1.5, "a"].map((v) => CD.of(v));
  assert(same(ofs, [CD.Int, T("long"), T("unsigned long"), CD.Bool, CD.Double, CD.String]));
  assert(CD.of(2n ** 64n) === D.Anything && CD.of(null) === D.Anything);
  assert(T("uint8_t").contains(255n) && !T("uint8_t").contains(256n) && T("int8_t").contains(-128n) && !T("int8_t").contains(-129n));
  assert(T("float").contains(1.5) && T("long double").contains("1.5") && CD.Bool.contains(true) && !CD.Int.contains(true));
  assert(!CD.Int.contains(1.5) && CD.Int.includes(T("int32_t")) && CD.Int.includes(new D.OfUnion(CD.Int)) && !CD.Int.includes(T("long")));
  const infer = (expression: unknown, environment?: Record<string, D.Domain>) => C.DIALECT.infer(expression, environment);
  assert(infer(C.binary("+", C.constant(1n, "uint8_t"), 1n)) === CD.Int && (infer(C.binary("+", 1n, C.constant(1n, "unsigned int"))) as any).name() === "unsigned int");
  assert(infer(C.binary("<<", C.constant(1n, "uint8_t"), C.constant(1n, "long"))) === CD.Int && infer(C.binary("<", 1n, 2.0)) === CD.Bool);
  assert(infer(C.binary("&&", x, 1n), { x: D.Anything }) === CD.Bool && infer(C.binary("+", x, 1n), { x: D.Anything }) === D.Anything);
  assert(T("uint16_t").equals(infer(C.cast("uint16_t", x), { x: D.Anything })) && T("uint8_t").equals(infer(C.constant(5n, "uint8_t"))));
  assert(infer(C.unary("~", C.constant(1n, "uint8_t"))) === CD.Int && infer(C.unary("-", 1.5)) === CD.Double && infer(C.unary("!", 1.5)) === CD.Bool);
  assert((infer(C.binary("%", C.constant(7n, "uint32_t"), C.constant(2n, "uint32_t"))) as any).name() === "unsigned int");
  assert(infer(C.conditional(x, 1n, 2n), { x: D.Anything }) === D.Anything && infer(C.member(x, "a"), { x: D.Anything }) === D.Anything);
  const refused: [unknown, string][] = [[C.binary("%", 1.5, 1n), "% cannot take (double, int); it takes (integer, integer) -> their common type"],
    [C.binary("+", "a", 1n), "+ cannot take (string, int)"], [C.binary("<<", 1.5, 1n), "<< cannot take (double, int)"],
    [C.unary("~", 1.5), "~ cannot take (double); it takes (integer) -> its promoted type"]];
  for (const [expression, message] of refused) raises(TypeError, () => infer(expression), message);
}

## DIA-12 · Ccpp evaluates by C's rules: promotions and conversions, wrapping unsigned arithmetic, and undefined behavior raising

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => CV.OfAny(expression, variables) as any;
  const x = C.identifier("x");
  const U = (value: bigint) => C.constant(value, "unsigned int");
  assert(ev(C.binary("+", U(4294967295n), 1n)) === 0n && ev(C.binary("-", U(0n), 1n)) === 4294967295n && ev(C.binary("*", U(65536n), U(65536n))) === 0n);
  assert(ev(C.binary("<", -1n, U(0n))) === false && ev(C.binary("<", -1n, C.constant(0n, "unsigned short"))) === true); // promoted to int
  assert(ev(C.binary("+", C.constant(200n, "uint8_t"), C.constant(100n, "uint8_t"))) === 300n); // in int
  assert(ev(C.binary("/", -7n, 2n)) === -3n && ev(C.binary("%", -7n, 2n)) === -1n && ev(C.binary("/", 7n, -2n)) === -3n && ev(C.binary("%", 7n, -2n)) === 1n);
  assert(ev(C.binary("/", 7.0, 2n)) === 3.5 && ev(C.binary("/", 1.0, 0.0)) === Infinity && Number.isNaN(ev(C.binary("/", 0.0, 0.0))));
  assert(ev(C.binary("+", C.cast("float", 1.5), C.cast("float", 0.1))) === 1.600000023841858 && ev(C.binary("-", 1.5, 1n)) === 0.5);
  assert(ev(C.binary("*", C.cast("long double", 1.5), 2n)) === "3");
  assert(ev(C.binary("&", 12n, 10n)) === 8n && ev(C.binary("^", 12n, 10n)) === 6n && ev(C.binary("|", -1n, U(0n))) === 4294967295n);
  assert(ev(C.binary("<<", U(1n), 31n)) === 2147483648n && ev(C.binary(">>", -8n, 1n)) === -4n && ev(C.binary("<<", C.constant(1n, "uint8_t"), 8n)) === 256n);
  assert(ev(C.unary("-", U(1n))) === 4294967295n && ev(C.unary("~", C.constant(0n, "uint8_t"))) === -1n && ev(C.unary("~", U(0n))) === 4294967295n);
  assert(ev(C.unary("+", true)) === 1n && ev(C.unary("-", 1.5)) === -1.5 && ev(C.unary("+", 1.5)) === 1.5);
  assert(ev(C.unary("!", 0.0)) === true && ev(C.unary("!", -0)) === true && ev(C.unary("!", NaN)) === false && ev(C.unary("!", 3n)) === false);
  assert(ev(C.binary("==", NaN, NaN)) === false && ev(C.binary("!=", NaN, 1.0)) === true && ev(C.binary("==", -0, 0.0)) === true);
  assert(ev(C.binary("<", 1n, 1.5)) === true && ev(C.binary(">=", C.cast("long double", 2n), 1n)) === true && ev(C.binary(">", 2n, 1n)) === true);
  assert(ev(C.binary("<=", Infinity, Infinity)) === true && ev(C.binary("<", C.cast("long double", NaN), 1n)) === false);
  assert(ev(C.binary("&&", 0n, C.binary("/", 1n, 0n))) === false && ev(C.binary("||", 2n, C.binary("/", 1n, 0n))) === true); // decided early
  assert(ev(C.binary("&&", 1n, 2.5)) === true && ev(C.binary("||", 0n, 0.0)) === false);
  assert(ev(C.conditional(C.binary(">", x, 0n), C.binary("/", 10n, x), 0n), { x: 0n }) === 0n && ev(C.conditional(1n, 2n, 3.5)) === 2n); // the chosen one
  assert(ev(C.cast("uint8_t", 300n)) === 44n && ev(C.cast("int8_t", 200n)) === -56n && ev(C.cast("int", 2.9)) === 2n && ev(C.cast("int", -2.9)) === -2n);
  assert(ev(C.cast("bool", 0.5)) === true && ev(C.cast("bool", 0n)) === false && ev(C.cast("bool", -0)) === false);
  assert(ev(C.cast("double", 2n ** 60n + 1n)) === 2 ** 60 && ev(C.cast("float", 0.1)) === 0.10000000149011612 && ev(C.cast("int", true)) === 1n);
  assert(ev(C.cast("long double", 0.1)) === "0.1000000000000000055511151231257827" && ev(C.cast("double", C.cast("float", 0.1))) === 0.10000000149011612);
  assert(ev(C.constant(5n, "uint8_t")) === 5n && ev(C.constant("hi")) === "hi");
  // What C leaves undefined throws.
  const undefinedBehavior: [unknown, Function, string][] = [
    [C.binary("+", 2147483647n, 1n), FE.OverflowError, "signed overflow in +: 2147483648 does not fit 'int'"],
    [C.unary("-", -2147483648n), FE.OverflowError, "signed overflow in -: 2147483648 does not fit 'int'"],
    [C.binary("/", -2147483648n, -1n), FE.OverflowError, "signed overflow in /: 2147483648 does not fit 'int'"],
    [C.binary("%", 1n, 0n), FE.ZeroDivisionError, "integer division by zero"],
    [C.binary("<<", 1n, 32n), ValueError, "shift by 32 is undefined for 'int'"], [C.binary(">>", 1n, -1n), ValueError, "shift by -1 is undefined for 'int'"],
    [C.binary("<<", -1n, 1n), ValueError, "left shift of a negative value is undefined"],
    [C.binary("<<", 1n, 31n), FE.OverflowError, "signed overflow in <<: 2147483648 does not fit 'int'"],
    [C.cast("uint8_t", 300.0), FE.OverflowError, "300.0 is out of range of 'uint8_t'"],
    [C.cast("int", Infinity), FE.OverflowError, "inf is out of range of 'int'"],
    [C.cast("int", C.cast("long double", 1e300)), FE.OverflowError, " is out of range of 'int'"],
    [C.cast("int", NaN), ValueError, "NaN has no integer value"],
    [C.binary("%", 1.5, 1n), TypeError, "invalid operands to binary % ('double' and 'double')"],
    [C.binary("<<", 1.5, 1n), TypeError, "invalid operands to binary << ('double' and 'int')"],
    [C.unary("~", 1.5), TypeError, "invalid argument type 'double' to unary expression"],
    [C.binary("+", "a", 1n), TypeError, "invalid operand to +: 'str'"], [C.unary("!", "a"), TypeError, "invalid operand to !: 'str'"],
  ];
  for (const [expression, error, message] of undefinedBehavior) raises(error, () => ev(expression), message);
  // A variable's type is its value's: a typed value's by its width and signedness.
  raises(FE.OverflowError, () => ev(x, { x: 2n ** 64n }), "integer 18446744073709551616 is too large for any type");
  raises(TypeError, () => ev(x, { x: new BD.Value(new BD.OfEnum.Data(["A"]), "A") }), "enum(A) has no C type");
  const [U8, F32] = [new BD.OfInteger.Data(8n, false), new BD.OfIeee754.Data("binary32")];
  assert(ev(C.binary("+", x, 1n), { x: new BD.Value(U8, 255n) }) === 256n && ev(C.cast("uint8_t", C.binary("+", x, 1n)), { x: new BD.Value(U8, 255n) }) === 0n);
  assert(ev(C.binary("+", x, x), { x: new BD.Value(F32, 1.5) }) === 3.0 && ev(C.binary("+", x, 1n), { x: 2n ** 40n }) === 2n ** 40n + 1n);
  assert(ev(C.binary("*", x, 2n), { x: new BD.Value(new BD.OfInteger.Data(16n), -5n) }) === -10n && ev(C.constant(5n, "word")) === 5n); // no type: its value's
  // Members, elements and calls.
  const Board = new S.OfObject.Builder().ref().properties(text("name"), (p) => p.name("grid").of(
    (t) => t.as_indexed((i) => i.of((t) => t.as_indexed((j) => j.of(new S.OfNative.Data(BigInt))))))).create();
  Proxies.register("Board", Board);
  const board = (Proxies.Builders as any).Board().name("b").grid([[1n, 2n], [3n]]).create();
  assert(ev(C.member(x, "age"), { x: ann }) === 30n && ev(C.member(x, "age", "->"), { x: { age: 5n } }) === 5n);
  assert(ev(C.member(x, "age"), { x: new Map([["age", 7n]]) }) === 7n);
  assert(ev(C.subscript(x, 1n), { x: [10n, 20n] }) === 20n && ev(C.subscript(C.subscript(C.member(x, "grid"), 1n), 0n), { x: board }) === 3n);
  const refused: [unknown, unknown, Function, string][] = [
    [C.member(x, "phone"), ann, KeyError, "no member named 'phone'"],
    [C.member(x, "a"), 1n, TypeError, "member reference base type 'int' is not a structure"],
    [C.subscript(x, 2n), [10n, 20n], ValueError, "index 2 is out of bounds of an array of 2"],
    [C.subscript(x, 0n), 1n, TypeError, "subscripted value is not an array: 'int'"],
    [C.subscript(x, 0.5), [1n], TypeError, "array subscript is not an integer: 'double'"],
    [C.call("f", 1n), null, FE.NameError, "use of undeclared identifier 'f'"], [C.identifier("z"), null, FE.NameError, "use of undeclared identifier 'z'"],
  ];
  for (const [expression, variable, error, message] of refused) raises(error, () => ev(expression, { x: variable }), message);
  assert(CV.OfAny(C.call("max", x, 3n), new CV.Scope({ x: 5n }, { functions: { max: (a: bigint, b: bigint) => (a > b ? a : b) } })) === 5n);
}

## DIA-13 · SystemVerilog: expressions over 4-state vectors and reals, with IEEE 1800's precedence, sized literals and casts

In [ ]:
{
  const [x, y, v] = [SV.identifier("x"), SV.identifier("y"), SV.vector];
  const cases: [unknown, string][] = [
    [SV.binary("+", x, SV.binary("*", y, 2n)), "x + y * 2"], [SV.binary("*", SV.binary("+", x, y), 2n), "(x + y) * 2"],
    [SV.binary("-", x, SV.binary("-", y, 1n)), "x - (y - 1)"], [SV.binary("**", SV.binary("**", x, y), 2n), "x ** y ** 2"],
    [SV.binary("**", x, SV.binary("**", y, 2n)), "x ** (y ** 2)"], [SV.binary("<<", SV.binary("+", x, 1n), 2n), "x + 1 << 2"],
    [SV.binary("&&", SV.binary("||", x, y), x), "(x || y) && x"], [SV.binary("||", SV.binary("&&", x, y), x), "x && y || x"],
    [SV.binary("^~", SV.binary("|", x, y), SV.binary("&", x, y)), "(x | y) ^~ x & y"],
    [SV.binary("==", SV.binary("&", x, 1n), y), "(x & 1) == y"], [SV.binary("<", SV.binary(">>>", x, 1n), SV.binary("%", y, 2n)), "x >>> 1 < y % 2"],
    [SV.binary("->", SV.binary("->", x, y), x), "(x -> y) -> x"], [SV.binary("<->", x, SV.binary("->", y, x)), "x <-> y -> x"],
    [SV.binary("->", SV.conditional(x, y, 1n), y), "x ? y : 1 -> y"], [SV.conditional(SV.binary("->", x, y), 1n, 2n), "(x -> y) ? 1 : 2"],
    [SV.conditional(x, y, SV.conditional(y, 1n, 2n)), "x ? y : y ? 1 : 2"], [SV.conditional(SV.conditional(x, y, 1n), 2n, 3n), "(x ? y : 1) ? 2 : 3"],
    [SV.unary("-", SV.unary("-", x)), "- -x"], [SV.unary("&", SV.unary("&", x)), "& &x"], [SV.unary("~", SV.unary("!", x)), "~ !x"],
    [SV.unary("-", -1n), "- -1"], [SV.binary("-", x, -1n), "x - -1"], [SV.unary("~&", SV.binary("+", x, 1n)), "~&(x + 1)"],
    [SV.inside(SV.binary("+", x, 1n), 1n, SV.span(2n, 5n)), "x + 1 inside {1, [2:5]}"], [SV.binary("<", SV.inside(x, 1n), 0n), "x inside {1} < 0"],
    [SV.binary("+", SV.inside(x, 1n), 1n), "(x inside {1}) + 1"], [SV.binary("==?", x, v("1x")), "x ==? 2'b1x"],
    [v("11111111", null, "h"), "8'hff"], [v("10x1", null, "h"), "4'b10x1"], [v("1xxxx", null, "h"), "5'h1x"], [v("1zzz", null, "h"), "4'b1zzz"],
    [v("00000101", true, "d"), "8'sd5"], [v("xxxx", null, "d"), "4'dx"], [v("0x", null, "d"), "2'b0x"], [v("111", null, "o"), "3'o7"],
    [v("10z1"), "4'b10z1"], [v("1", true), "1'sb1"],
    [SV.constant(1.5), "1.5"], [SV.constant(1.0), "1.0"], [SV.constant(1e300), "1e+300"], [SV.constant(NaN), "0.0 / 0.0"],
    [SV.constant(Infinity), "1.0 / 0.0"], [SV.constant(-Infinity), "-1.0 / 0.0"], [SV.binary("*", x, Infinity), "x * (1.0 / 0.0)"],
    [SV.unary("-", Infinity), "-(1.0 / 0.0)"], [SV.constant('a"\\\n\t'), '"a\\"\\\\\\n\\t"'], [SV.constant(-5n), "-5"],
    [SV.cast("signed", x), "signed'(x)"], [SV.cast(8n, SV.binary("+", x, 1n)), "8'(x + 1)"], [SV.cast("byte", -1n), "byte'(-1)"],
    [SV.member(SV.member(x, "a"), "b"), "x.a.b"], [SV.select(SV.binary("+", x, y), 0n), "(x + y)[0]"], [SV.range_(x, 7n, 4n), "x[7:4]"],
    [SV.concatenation(x, SV.replication(3n, y)), "{x, {3{y}}}"], [SV.call("$clog2", x), "$clog2(x)"], [SV.call("f", x, 1n), "f(x, 1)"],
  ];
  for (const [expression, source] of cases) assert(SVText.ToText(expression) === source, `${source}: ${SVText.ToText(expression)}`);
  // Validation: a vector's bits and base, and a cast's type or width.
  assert(same(v("12").validate(), ["a vector's bits must be 0, 1, x and z, at least one, got '12'"]));
  assert(same(v("").validate(), ["a vector's bits must be 0, 1, x and z, at least one, got ''"]));
  assert(same(v("1", null, "q").validate(), ["a vector's base must be b, o, d or h, got 'q'"]));
  assert(same(SV.cast("word", x).validate({ bound: ["x"] }), ["a cast's type must be a built-in type, signed or unsigned, got 'word'"]));
  assert(same(SV.cast(0n, x).validate({ bound: ["x"] }), ["a cast's width must be positive, got 0"]));
  assert(same(new (kindOf(SV.DIALECT, "cast"))(null, null, x).validate({ bound: ["x"] }), ["a cast names a type or a width, not both nor neither"]));
  assert(same(SV.binary("<<=", x, 1n).validate({ bound: ["x"], core: true }), ["'<<=' is not a core operation"]));
  assert(SV.call("f", x).validate({ bound: ["x"], core: true }).length === 0 && same(SV.identifier("x").validate(), ["identifier 'x' is not bound"]));
  // Types: integral vectors of a width, a signedness and 2 or 4 states, and reals.
  const T = (name: string) => SVD.TYPES.get(name) as SystemVerilog.Domains.SvType;
  const widths = ["bit", "logic", "reg", "byte", "shortint", "int", "longint", "integer", "time"].map((n) => T(n).width);
  assert(same(widths, [1, 1, 1, 8, 16, 32, 64, 32, 64]));
  assert(T("int").signed && T("int").states === 2 && T("integer").states === 4 && !T("time").signed && T("logic").equals(T("reg")));
  assert(T("int").name() === "int" && String(T("integer")) === "integer" && SVD.vector(8).name() === "logic [7:0]");
  assert(SVD.vector(4, true, 2).name() === "bit signed [3:0]" && SVD.vector(32, true, 2).equals(T("int")) && T("shortreal").format === "binary32");
  assert(SVD.common([T("byte"), T("int")]).equals(T("int")) && SVD.common([T("byte"), SVD.vector(16)]).equals(SVD.vector(16)));
  assert(SVD.common([T("shortreal"), T("int")]).equals(T("real")) && SVD.common([T("shortreal"), T("shortreal")]).equals(T("shortreal")));
  assert(SVD.common([T("bit"), T("logic")]).equals(T("logic")));
  const ofs = [1n, -(2n ** 31n), 2n ** 31n, 2n ** 63n, 1.5, "a", true].map((value) => SVD.of(value));
  const expectedOfs = [T("integer"), T("integer"), T("longint"), D.Anything, T("real"), D.Anything, D.Anything];
  assert(ofs.every((domain, i) => domain === expectedOfs[i] || (expectedOfs[i] as any).equals(domain)));
  // Values: a Logic is a type and its bits, encoded as VPI does.
  const a = SVD.Logic.of("10xz");
  assert(a.aval === 0b1010n && a.bval === 0b0011n && a.bits === "10xz" && !a.known && String(a) === "4'b10xz");
  assert(SVD.Logic.of("10xz", true, 2).bits === "1000" && String(SVD.Logic.of("1", true)) === "1'sb1");
  assert(SVD.Logic.of("1110", true).integer() === -2n && SVD.Logic.of("1110").integer() === 14n && SVD.Logic.number(T("byte"), -1n).bits === "11111111");
  raises(ValueError, () => a.integer(), "4'b10xz has unknown bits");
  assert(T("int").contains(SVD.Logic.number(T("int"), 5n)) && !T("int").contains(SVD.Logic.number(T("integer"), 5n)) && !T("int").contains(5n));
  assert(T("real").contains(1.5) && !T("real").contains(1n) && T("int").includes(SVD.vector(32, true, 2)) && !T("int").includes(T("integer")));
  assert(T("int").includes(new D.OfUnion(T("int"))) && !T("int").includes(D.Anything));
  // Inference sizes an expression as IEEE 1800 does.
  const infer = (expression: unknown, environment?: Record<string, any>) => SV.DIALECT.infer(expression, environment) as any;
  const is = (domain: unknown, expected: SystemVerilog.Domains.SvType) => expected.equals(domain);
  assert(is(infer(SV.binary("+", v("1010"), 1n)), SVD.vector(32)) && is(infer(SV.binary("+", 1n, 2n)), T("integer")));
  assert(is(infer(SV.binary("+", SV.cast("byte", 1n), SV.cast("int", 1n))), T("int")) && is(infer(SV.binary("*", 1n, 1.5)), T("real")));
  assert(is(infer(SV.binary("**", v("1010"), 2n)), SVD.vector(4)) && is(infer(SV.binary("**", 2n, 0.5)), T("real")));
  assert(is(infer(SV.binary("<<", v("1010"), 40n)), SVD.vector(4)) && is(infer(SV.binary("==", v("1010"), 1n)), SVD.vector(1)));
  assert(is(infer(SV.binary("===", v("1010"), 1n)), T("bit")) && is(infer(SV.binary("==?", v("1010"), 1n)), SVD.vector(1)));
  assert(is(infer(SV.binary("&&", x, 1n), { x: D.Anything }), SVD.vector(1)) && infer(SV.binary("+", x, 1n), { x: D.Anything }) === D.Anything);
  assert(is(infer(SV.binary("&", v("10"), v("1000"))), SVD.vector(4)) && is(infer(SV.unary("~", SV.cast("byte", 1n))), T("byte")));
  assert(is(infer(SV.unary("-", 1.5)), T("real")) && is(infer(SV.unary("!", 1.5)), SVD.vector(1)) && is(infer(SV.unary("^", v("10"))), SVD.vector(1)));
  assert(is(infer(SV.binary("%", 7n, 2n)), T("integer")) && is(infer(SV.cast("int", x), { x: D.Anything }), T("int")));
  assert(infer(SV.cast(8n, x), { x: D.Anything }) === D.Anything && infer(SV.cast("signed", 1n)) === D.Anything); // sized by the operand
  assert(is(infer(SV.inside(x, 1n), { x: D.Anything }), SVD.vector(1)) && infer(SV.conditional(x, 1n, 2n), { x: D.Anything }) === D.Anything);
  assert(v("12").typed() === null && is(infer(v("10x1", true)), SVD.vector(4, true)) && is(infer(SV.constant(1.5)), T("real")));
  assert(infer(SV.binary("+", "a", 1n)) === D.Anything); // a string has no type of its own
  raises(TypeError, () => infer(SV.binary("+", x, 1n), { x: new D.OfUnion(T("int"), T("real")) }),
    "+ cannot take (int | real, integer); it takes (T, T) -> their common type");
  assert(SVD.BINARY.get("+")?.describe() === "(T, T) -> their common type" && SVD.UNARY.get("!")?.arity() === 1);
  const refused: [unknown, string][] = [
    [SV.binary("%", 1.5, 1n), "% cannot take (real, integer); it takes (integral, integral) -> their common type"],
    [SV.binary("&", 1.5, 1n), "& cannot take (real, integer)"], [SV.binary("<<", 1.5, 1n), "<< cannot take (real, integer)"],
    [SV.binary("===", 1.5, 1n), "=== cannot take (real, integer)"], [SV.binary("==?", 1.5, 1n), "==? cannot take (real, integer)"],
    [SV.unary("~", 1.5), "~ cannot take (real); it takes (integral) -> its type"], [SV.unary("&", 1.5), "& cannot take (real)"],
  ];
  for (const [expression, message] of refused) raises(TypeError, () => infer(expression), message);
}

## DIA-14 · SystemVerilog evaluates by IEEE 1800's rules: sizing by context, 4-state logic, casts and system functions

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => SVV.OfAny(expression, variables) as any;
  const [v, b, u, c, call] = [SV.vector, SV.binary, SV.unary, SV.cast, SV.call];
  const x = SV.identifier("x");
  const T = (name: string) => SVD.TYPES.get(name) as SystemVerilog.Domains.SvType;
  // Each expression and its value: a Logic by its text, or a real.
  const cases: [unknown, string | number][] = [
    [b("+", v("1010"), v("0110")), "4'b0000"], [b("+", v("1111"), 1n), "32'b" + "0".repeat(27) + "10000"],
    [b("/", -7n, 2n), "32'sb" + "1".repeat(30) + "01"], [b("%", -7n, 2n), "32'sb" + "1".repeat(32)],
    [b("/", 1n, 0n), "32'sb" + "x".repeat(32)], [b(">>>", v("11111110", true), 1n), "8'sb11111111"], [b(">>>", v("1000"), 1n), "4'b0100"],
    [b(">>", v("1000", true), 1n), "4'sb0100"], [b("&", v("10x1"), v("0011")), "4'b00x1"], [b("|", v("10z1"), v("1100")), "4'b11x1"],
    [b("^", v("10x1"), v("0011")), "4'b10x0"], [b("~^", v("1001"), v("0011")), "4'b0101"], [b("+", v("10x1"), 1n), "32'b" + "x".repeat(32)],
    [b("==", v("10x1"), v("10x1")), "1'bx"], [b("===", v("10x1"), v("10x1")), "1'b1"], [b("!==", v("10z1"), v("10x1")), "1'b1"],
    [b("==?", v("1001"), v("1xx1")), "1'b1"], [b("!=?", v("1001"), v("1xx0")), "1'b1"], [b("==?", v("1x01"), v("1001")), "1'bx"],
    [u("&", v("1111")), "1'b1"], [u("|", v("000x")), "1'bx"], [u("^", v("1011")), "1'b1"], [u("~&", v("1101")), "1'b1"],
    [u("~|", v("0000")), "1'b1"], [u("&", v("1x11")), "1'bx"], [u("|", v("0100")), "1'b1"], [u("~^", v("1x11")), "1'bx"], [u("~^", v("1011")), "1'b0"], [u("^", v("1x11")), "1'bx"], [u("&", v("0x11")), "1'b0"],
    [u("-", v("0001")), "4'b1111"], [u("~", v("10x1")), "4'b01x0"], [u("-", v("0x01")), "4'bxxxx"], [u("+", v("0x01")), "4'b0x01"],
    [b("**", 3n, -1n), "32'sb" + "0".repeat(32)], [b("**", 0n, -1n), "32'sb" + "x".repeat(32)],
    [b("**", -1n, -1n), "32'sb" + "1".repeat(32)], [b("**", -1n, -2n), "32'sb" + "0".repeat(31) + "1"],
    [b("**", 2n, 10n), "32'sb" + "0".repeat(21) + "10000000000"], [b("**", 1n, -3n), "32'sb" + "0".repeat(31) + "1"],
    [b("**", 2.0, 0.5), 2 ** 0.5], [b("/", 1.0, 0n), Infinity], [b("**", 2n, 0.5), 2 ** 0.5], [b("+", 1n, 0.5), 1.5],
    [SV.conditional(v("x"), v("1100"), v("1010")), "4'b1xx0"], [SV.conditional(v("1"), v("1100"), v("1010")), "4'b1100"],
    [SV.conditional(0n, v("1100"), v("1010")), "4'b1010"], [SV.conditional(v("z"), 1.5, 1.5), 1.5], [SV.conditional(v("z"), 1.5, 2.5), 0.0],
    [SV.inside(3n, 1n, SV.span(2n, 5n)), "1'b1"], [SV.inside(7n, 1n, SV.span(2n, 5n)), "1'b0"], [SV.inside(v("1x00"), v("1100")), "1'bx"],
    [SV.inside(v("1x00"), v("1100"), v("1x00")), "1'b1"], [SV.inside(v("1x00"), SV.span(0n, 3n)), "1'bx"],
    [SV.inside(v("1x00"), SV.span(8n, 15n)), "1'bx"], [SV.inside(v("1100"), v("1x00")), "1'b1"], [c("byte", 300n), "8'sb00101100"],
    [c("byte", 200n), "8'sb11001000"], [c("unsigned", -1n), "32'b" + "1".repeat(32)], [c(4n, v("11111111")), "4'b1111"],
    [c("int", 2.5), "32'sb" + "0".repeat(30) + "11"], [c("int", -2.5), "32'sb" + "1".repeat(30) + "01"], [c("real", 3n), 3.0],
    [c("shortreal", 0.1), 0.10000000149011612], [c("int", v("10x1")), "32'sb" + "0".repeat(28) + "1001"], [c("bit", v("x")), "1'b0"],
    [c("signed", v("1111")), "4'sb1111"], [c(8n, v("1111", true)), "8'sb11111111"], [c(8n, v("z1", true)), "8'sbzzzzzzz1"],
    [c(4n, v("x0", true)), "4'sbxxx0"], [call("$clog2", 8n), "32'sb" + "0".repeat(30) + "11"],
    [call("$clog2", 9n), "32'sb" + "0".repeat(29) + "100"], [call("$clog2", 0n), "32'sb" + "0".repeat(32)],
    [call("$clog2", v("1x")), "32'sb" + "x".repeat(32)], [call("$bits", v("00000000")), "32'sb" + "0".repeat(28) + "1000"],
    [call("$bits", 1.5), "32'sb" + "0".repeat(25) + "1000000"], [call("$countones", v("1011")), "32'sb" + "0".repeat(30) + "11"],
    [call("$onehot", v("0100")), "1'b1"], [call("$onehot", v("0000")), "1'b0"], [call("$onehot0", v("0000")), "1'b1"],
    [call("$isunknown", v("10x1")), "1'b1"], [call("$isunknown", v("1001")), "1'b0"], [call("$signed", v("1111")), "4'sb1111"],
    [call("$unsigned", v("1111", true)), "4'b1111"], [SV.concatenation(v("1010"), v("11")), "6'b101011"],
    [SV.replication(2n, v("10")), "4'b1010"], [SV.concatenation(v("x"), v("1")), "2'bx1"], [SV.select(v("1010"), 1n), "1'b1"],
    [SV.select(v("1010"), 4n), "1'bx"], [SV.select(v("1010"), v("x")), "1'bx"], [SV.range_(v("1010"), 3n, 2n), "2'b10"],
    [SV.range_(v("1010"), 5n, 2n), "4'bxx10"], [SV.select(c("int", v("1010")), 40n), "1'b0"],
    [b("+", v("1111", true), v("00000000", true)), "8'sb11111111"], [b("+", v("1111", true), v("00000000")), "8'b00001111"],
    [b("<", -1n, v("1")), "1'b0"], [b("<", -1n, 1n), "1'b1"], [b("==", b("+", v("1111"), v("0001")), v("10000")), "1'b1"],
    [b("==", b("<<", v("1000"), 1n), v("10000")), "1'b1"], [b("<<", v("0001"), v("10")), "4'b0100"],
    [b("<<", v("0001"), v("x")), "4'bxxxx"], [b("+", b("<", 2n, 1n), v("0001")), "4'b0001"], [b("+", b("<", 1n, 2n), v("0001")), "4'b0010"],
    [u("!", v("0000")), "1'b1"], [u("!", v("00x0")), "1'bx"], [u("!", 1.5), "1'b0"], [u("!", 0.0), "1'b1"], [b(">>>", v("x000", true), 1n), "4'sbxx00"], [b("&&", v("x"), 0n), "1'b0"],
    [b("||", v("x"), 1n), "1'b1"], [b("&&", v("x"), 1n), "1'bx"], [b("||", 0n, 0n), "1'b0"], [b("&&", 0n, v("x")), "1'b0"],
    [b("||", 1n, v("x")), "1'b1"], [b("->", 0n, v("x")), "1'b1"], [b("->", 1n, v("x")), "1'bx"], [b("->", 1n, 0n), "1'b0"],
    [b("->", v("x"), 1n), "1'b1"], [b("<->", 1n, 1n), "1'b1"], [b("<->", 1n, 0n), "1'b0"], [b("<->", v("x"), 0n), "1'bx"],
    [b(">", 1.5, 1n), "1'b1"], [b("==", NaN, NaN), "1'b0"], [b("!=", NaN, 1.0), "1'b1"], [b("<", v("x"), 1n), "1'bx"], [u("-", 1.5), -1.5],
    [u("+", 1.5), 1.5], [b("-", c("shortreal", 1.5), c("shortreal", 0.25)), 1.25], [b("*", 3n, 4n), "32'sb" + "0".repeat(28) + "1100"],
    [b("-", 3n, 4n), "32'sb" + "1".repeat(32)], [b("%", 7n, -2n), "32'sb" + "0".repeat(31) + "1"],
  ];
  for (const [expression, value] of cases) {
    const result = ev(expression);
    assert(typeof value === "string" ? String(result) === value : result === value, `${SVText.ToText(expression)}: ${result}`);
  }
  assert(Number.isNaN(ev(b("**", -8.0, 1 / 3))) && ev(b("**", 0.0, -1.0)) === Infinity && ev(b("**", 10.0, 400.0)) === Infinity);
  assert(ev(b("**", -10.0, 401.0)) === -Infinity && ev(b("**", -10.0, 400.0)) === Infinity && ev(b("**", c("shortreal", 2n), c("shortreal", 0.5))) === 1.4142135381698608);
  assert(ev(b("**", -0, -1.0)) === -Infinity && ev(b("**", -0, -2.0)) === Infinity && ev(b("**", 0.0, -3.0)) === Infinity);
  assert(ev(b("**", 1.0, NaN)) === 1.0 && ev(b("**", -1.0, Infinity)) === 1.0 && ev(b("**", NaN, 0.0)) === 1.0); // as C's pow
  assert(String(ev(b("/", c("int", 1n), c("int", 0n)))) === "32'sb" + "0".repeat(32) && String(ev(b("&", c("int", 5n), v("x")))) === "32'b" + "0".repeat(31) + "x");
  // Variables are typed by their values, Basic's typed values among them.
  assert(String(ev(x, { x: true })) === "1'b1" && ev(x, { x: 5n }).type.equals(T("integer")) && ev(x, { x: 2n ** 40n }).type.equals(T("longint")));
  assert(String(ev(b("<<", 1n, v("1".repeat(64))))) === "32'sb" + "0".repeat(32) && String(ev(b(">>>", -4n, v("1".repeat(64))))) === "32'sb" + "1".repeat(32));
  assert(String(ev(call("$bits", c("shortreal", 1.5)))) === "32'sb" + "0".repeat(26) + "100000" && String(ev(x, { x: "é" })) === "16'b1100001110101001");
  assert(String(ev(x, { x: "AB" })) === "16'b0100000101000010" && String(ev(x, { x: "" })) === "8'b00000000" && ev(x, { x: 1.5 }) === 1.5);
  assert(String(ev(x, { x: SVD.Logic.of("1x") })) === "2'b1x"
    && String(ev(b("+", x, 1n), { x: new BD.Value(new BD.OfInteger.Data(8n, false), 200n) })) === "32'b" + "0".repeat(24) + "11001001");
  assert(ev(x, { x: new BD.Value(new BD.OfInteger.Data(4n), -1n) }).type.equals(SVD.vector(4, true, 2))
    && String(ev(x, { x: new BD.Value(new BD.OfBits.Data(12n), new Uint8Array([0x0a, 0xbc])) })) === "12'b101010111100");
  assert(String(ev(x, { x: new BD.Value(new BD.OfIeee1164.Data(4n), "10ZU") })) === "4'b10zx"); // a std_logic_vector is a logic vector
  const levels = [..."01ZLHUXW-"].map((level) => String(ev(x, { x: new BD.Value(new BD.OfIeee1164.Data(), level) })));
  assert(same(levels, ["1'b0", "1'b1", "1'bz", "1'b0", "1'b1", "1'bx", "1'bx", "1'bx", "1'bx"]));
  const record = new Map([["a", 1n]]);
  assert(ev(x, { x: new BD.Value(new BD.OfIeee754.Data("binary32"), 1.5) }) === 1.5 && ev(x, { x: record }) === record);
  assert(String(ev(SV.member(x, "age"), { x: ann })) === "32'sb" + "0".repeat(26) + "011110" && String(ev(SV.member(x, "a"), { x: { a: true } })) === "1'b1");
  assert(String(ev(SV.select(x, 1n), { x: [1n, 2n] })) === "32'sb" + "0".repeat(30) + "10"
    && String(ev(SV.select(SV.member(x, "a"), 0n), { x: new Map([["a", [false]]]) })) === "1'b0");
  const Bus = new S.OfObject.Builder().ref().properties(text("name"), (p) => p.name("ports").of(
    (t) => t.as_indexed((i) => i.of(new S.OfNative.Data(BigInt))))).create();
  Proxies.register("Bus", Bus);
  const bus = (Proxies.Builders as any).Bus().name("b").ports([4n, 5n]).create();
  assert(String(ev(SV.select(SV.member(x, "ports"), 1n), { x: bus })) === "32'sb" + "0".repeat(29) + "101");
  const f = (a: any, b: any) => a.integer() + b.integer();
  assert((SVV.OfAny(SV.call("f", x, 1n), new SVV.Scope({ x: 2n }, { functions: { f } })) as any).type.equals(T("integer")));
  assert((SVV.OfAny(b("+", 1n, 1n), new SVV.Scope()) as any).integer() === 2n && (SVV.OfAny(SV.binary("+", 1n, 2n)) as any).integer() === 3n);
  const errors: [[unknown, Record<string, unknown>], Function, string][] = [
    [[x, {}], NameError, "identifier 'x' is not declared"], [[SV.call("f", 1n), {}], NameError, "function 'f' is not declared"],
    [[v("12"), {}], ValueError, "cannot evaluate an invalid expression: a vector's bits must be 0, 1, x and z, at least one, got '12'"],
    [[x, { x: 2n ** 70n }], OverflowError, "integer 1180591620717411303424 is too large for any type"],
    [[x, { x: new BD.Value(new BD.OfIeee754.Data("decimal64"), "1") }], TypeError, "decimal64 has no SystemVerilog type"],
    [[x, { x: new BD.Value(new BD.OfInteger.Data(), 1n) }], TypeError, "int has no SystemVerilog type"],
    [[u("~", 1.5), {}], TypeError, "~ expects an integral operand, got real"],
    [[u("&", 1.5), {}], TypeError, "the reduction & expects an integral operand, got real"],
    [[b("%", 1.5, 1n), {}], TypeError, "% expects integral operands, got real"],
    [[b("&", 1.5, 1n), {}], TypeError, "& expects integral operands, got real"],
    [[b("<<", 1.5, 1n), {}], TypeError, "<< expects an integral operand, got real"],
    [[b("<<", 1n, 1.5), {}], TypeError, "<< expects integral operands, got real"],
    [[b("+", x, 1n), { x: new Map() }], TypeError, "+ expects a number, got dict"],
    [[c("int", x), { x: new Map() }], TypeError, "a cast expects a number, got dict"],
    [[SV.conditional(1n, x, 1n), { x: new Map() }], TypeError, "?: expects a number, got dict"],
    [[u("!", x), { x: new Map() }], TypeError, "a dict has no truth value"],
    [[c("signed", 1.5), {}], TypeError, "signed' expects an integral operand, got real"],
    [[c("int", NaN), {}], OverflowError, "nan has no integer value"],
    [[c("int", -Infinity), {}], OverflowError, "-inf has no integer value"],
    [[SV.member(x, "b"), { x: new Map([["a", 1n]]) }], KeyError, "no member named 'b'"],
    [[SV.member(x, "b"), { x: 1n }], TypeError, "a Logic has no members"],
    [[SV.select(x, 2n), { x: [1n, 2n] }], ValueError, "index 2 is out of bounds of an array of 2"],
    [[SV.select(x, v("x")), { x: [1n, 2n] }], ValueError, "an array's index must be known, got 1'bx"],
    [[SV.select(x, 0n), { x: new Map([["a", 1n]]) }], TypeError, "a dict cannot be selected from"],
    [[SV.range_(x, 1n, 0n), { x: new Map() }], TypeError, "a dict cannot be selected from"],
    [[SV.range_(v("1010"), 0n, 1n), {}], ValueError, "a range's msb must be at least its lsb, got [0:1]"],
    [[SV.range_(v("1010"), v("x"), 0n), {}], ValueError, "a range's msb must be a known integer, got 1'bx"],
    [[SV.range_(v("1010"), 1n, 0.5), {}], ValueError, "a range's lsb must be a known integer, got 0.5"],
    [[SV.replication(0n, v("1")), {}], ValueError, "a replication's count must be positive, got 0"],
    [[SV.concatenation(v("1"), 1.5), {}], TypeError, "a concatenation's parts must be integral, got float"],
    [[call("$clog2", 1n, 2n), {}], TypeError, "$clog2 takes 1 argument, got 2"],
    [[call("$signed", 1.5), {}], TypeError, "$signed expects an integral argument, got real"],
    [[call("$bits", x), { x: new Map() }], TypeError, "$bits expects a number, got dict"],
    [[call("$countones", 1.5), {}], TypeError, "$countones expects an integral argument, got real"],
  ];
  for (const [[expression, variables], error, message] of errors) raises(error, () => ev(expression, variables), message);
}

## DIA-15 · SystemVerilog arrays: methods, reductions with a `with` clause, and `inside` over an array's elements

In [ ]:
{
  const [xs, p, q] = [SV.identifier("xs"), SV.identifier("p"), SV.identifier("q")];
  assert(SVText.ToText(SV.iterate(SV.member(xs, "pins"), "and", "p", SV.binary(">", p, 0n))) === "xs.pins.and(p) with (p > 0)");
  assert(SVText.ToText(SV.select(SV.method(xs, "min"), 0n)) === "xs.min()[0]" && SVText.ToText(SV.method(SV.binary("+", xs, 1n), "size")) === "(xs + 1).size()");
  assert(SVText.ToText(SV.inside(1n, xs, SV.span(2n, 3n))) === "1 inside {xs, [2:3]}");
  assert(same(SV.method(xs, "pop").validate({ bound: ["xs"] }), [
    "an array method must be one of size, sum, product, and, or, xor, min, max, unique, got 'pop'"]));
  assert(same(SV.iterate(xs, "min", "p", p).validate({ bound: ["xs"] }), [ // a malformed iteration binds no name
    "an iteration's method must be one of sum, product, and, or, xor, got 'min'", "body: identifier 'p' is not bound"]));
  assert(SV.iterate(xs, "and", "p", p).validate({ bound: ["xs"] }).length === 0
    && same([...SY.free(SV.iterate(xs, "and", "p", SV.binary("&", p, q)))].sort(), ["q", "xs"]));
  assert(SV.DIALECT.infer(SV.iterate(xs, "and", "p", p), { xs: D.Anything }) === D.Anything);
  assert(SV.DIALECT.infer(SV.method(xs, "size"), { xs: D.Anything }) === D.Anything);
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => SVV.OfAny(expression, variables) as any;
  const data = [1n, 2n, 3n, 2n];
  const cases: [unknown, string][] = [ // each over [1, 2, 3, 2], and its value
    [SV.method(xs, "size"), "32'sb" + "0".repeat(29) + "100"], [SV.method(xs, "sum"), "32'sb" + "0".repeat(28) + "1000"],
    [SV.method(xs, "product"), "32'sb" + "0".repeat(28) + "1100"], [SV.method(xs, "and"), "32'sb" + "0".repeat(32)],
    [SV.method(xs, "or"), "32'sb" + "0".repeat(30) + "11"], [SV.method(xs, "xor"), "32'sb" + "0".repeat(30) + "10"],
    [SV.select(SV.method(xs, "min"), 0n), "32'sb" + "0".repeat(31) + "1"], [SV.select(SV.method(xs, "max"), 0n), "32'sb" + "0".repeat(30) + "11"],
    [SV.method(SV.method(xs, "unique"), "size"), "32'sb" + "0".repeat(30) + "11"],
    [SV.iterate(xs, "and", "p", SV.binary(">", p, 0n)), "1'b1"], [SV.iterate(xs, "and", "p", SV.binary(">", p, 1n)), "1'b0"],
    [SV.iterate(xs, "or", "p", SV.binary(">", p, 2n)), "1'b1"], [SV.iterate(xs, "xor", "p", SV.binary("==", p, 2n)), "1'b0"],
    [SV.iterate(xs, "sum", "p", SV.cast("int", SV.binary(">=", p, 2n))), "32'sb" + "0".repeat(30) + "11"],
    [SV.iterate(xs, "sum", "p", SV.binary("*", p, p)), "32'sb" + "0".repeat(27) + "10010"],
    [SV.iterate(xs, "product", "p", SV.binary("+", p, 1n)), "32'sb" + "0".repeat(25) + "1001000"],
    [SV.inside(3n, xs), "1'b1"], [SV.inside(5n, xs, 7n), "1'b0"], [SV.inside(7n, xs, 7n), "1'b1"], [SV.inside(2n, SV.span(0n, 1n), xs), "1'b1"],
  ];
  for (const [expression, value] of cases) assert(String(ev(expression, { xs: data })) === value, SVText.ToText(expression));
  // x and z reduce as the operators take them: and() is 0 if any value is, x if any is unknown and none 0.
  assert(String(ev(SV.iterate(xs, "and", "p", p), { xs: [SVD.Logic.of("1"), SVD.Logic.of("x")] })) === "1'bx");
  assert(String(ev(SV.iterate(xs, "and", "p", p), { xs: [SVD.Logic.of("0"), SVD.Logic.of("x")] })) === "1'b0");
  assert(String(ev(SV.inside(SV.vector("1x"), xs), { xs: [SVD.Logic.of("10")] })) === "1'bx");
  assert(String(ev(SV.method(SV.method(xs, "unique"), "size"), { xs: [SVD.Logic.of("1x"), SVD.Logic.of("1x"), 1.0, 1n, 1.0] })) === "32'sb" + "0".repeat(30) + "11");
  // Of no items, the reductions give their identities, and the locators no item.
  assert(same(["sum", "product", "and", "or", "xor"].map((m) => String(ev(SV.method(xs, m), { xs: [] }))), [
    "32'sb" + "0".repeat(32), "32'sb" + "0".repeat(31) + "1", "1'b1", "1'b0", "1'b0"]));
  assert(ev(SV.method(xs, "min"), { xs: [] }).length === 0 && String(ev(SV.iterate(xs, "and", "p", SV.binary(">", p, 0n)), { xs: [] })) === "1'b1");
  assert(ev(SV.method(xs, "sum"), { xs: [1.5, 2n] }) === 3.5 && String(ev(SV.select(SV.method(xs, "max"), 0n), { xs: [1.5, 2n] })) === "32'sb" + "0".repeat(30) + "10");
  const errors: [unknown, Record<string, unknown>, Function, string][] = [
    [SV.method(xs, "size"), { xs: 5n }, TypeError, "size() is a method of arrays, not of Logic"],
    [SV.iterate(xs, "and", "p", p), { xs: new Map([["a", 1n]]) }, TypeError, "and() is a method of arrays, not of dict"],
    [SV.method(xs, "sum"), { xs: [new Map([["a", 1n]])] }, TypeError, "an array's elements must be numbers, got dict"],
    [SV.inside(1n, xs), { xs: [new Map([["a", 1n]])] }, TypeError, "an array's elements must be numbers, got dict"],
    [SV.inside(1n, xs), { xs: new Map([["a", 1n]]) }, TypeError, "inside expects a number, got dict"],
    [SV.method(xs, "and"), { xs: [1.5, 2.5] }, TypeError, "& expects integral operands, got real"],
  ];
  for (const [expression, variables, error, message] of errors) raises(error, () => ev(expression, variables), message);
}

## DIA-16 · Matlab arrays: arrayfun, indexing from 1, and the functions of arrays

In [ ]:
{
  const [xs, p] = [M.identifier("xs"), M.identifier("p")];
  assert(MText.ToText(M.call("all", M.arrayfun("p", M.field(xs, "pins"), M.binary(">", p, 0n)))) === "all(arrayfun(@(p) p > 0, xs.pins))");
  assert(MText.ToText(M.index(xs, M.binary("+", p, 1n))) === "xs(p + 1)" && MText.ToText(M.index(M.field(xs, "a"), 2n)) === "xs.a(2)");
  assert(same(M.arrayfun("_p", xs, 1n).validate({ bound: ["xs"] }), ["an arrayfun's parameter must be an identifier, got '_p'"]));
  assert(M.arrayfun("p", xs, p).validate({ bound: ["xs"] }).length === 0
    && same([...SY.free(M.arrayfun("p", xs, M.binary("+", p, M.identifier("q"))))].sort(), ["q", "xs"]));
  assert(same(M.call("numel", xs, 1n).validate({ bound: ["xs"] }), ["numel takes 1 arguments, got 2"]));
  assert(M.DIALECT.infer(M.call("all", xs), { xs: D.Anything }).name() === "logical");
  assert(M.DIALECT.infer(M.arrayfun("p", xs, p), { xs: D.Anything }) === D.Anything);
  const alike = (a: any, b: any): boolean => (Array.isArray(b)
    ? Array.isArray(a) && a.length === b.length && a.every((x, i) => alike(x, b[i]))
    : typeof a === typeof b && (a === b || (Number.isNaN(a) && Number.isNaN(b))));
  const VALUES: [any, unknown, unknown][] = [
    [M.call("all", M.arrayfun("p", xs, M.binary(">", p, 0n))), [1n, 2n, 3n, 2n], true],
    [M.call("any", M.arrayfun("p", xs, M.binary(">", p, 2n))), [1n, 2n, 3n, 2n], true],
    [M.call("nnz", M.arrayfun("p", xs, M.binary(">=", p, 2n))), [1n, 2n, 3n, 2n], 3.0], [M.call("numel", xs), [1n, 2n, 3n, 2n], 4.0],
    [M.index(xs, 2n), [1n, 2n, 3n, 2n], 2.0], [M.index(xs, true), [1n, 2n, 3n, 2n], 1.0], [M.index(xs, false), [1n, 2n, 3n, 2n], []],
    [M.call("ismember", 3n, xs), [1n, 2n, 3n, 2n], true], [M.call("ismember", "3", xs), [1n, 2n, 3n, 2n], true],
    [M.call("ismember", 5n, xs), [1n, 2n, 3n, 2n], false], [M.call("sum", xs), [1n, 2n, 3n, 2n], 8.0],
    [M.call("min", xs), [1n, 2n, 3n, 2n], 1.0], [M.call("max", xs), [1n, 2n, 3n, 2n], 3.0],
    [M.call("numel", M.call("unique", xs)), [1n, 2n, 3n, 2n], 3.0],
    [M.arrayfun("p", xs, M.binary(".*", p, 2n)), [1n, 2n, 3n, 2n], [2.0, 4.0, 6.0, 4.0]],
    [M.arrayfun("p", xs, M.binary(">", p, 1n)), [1n, 2n, 3n, 2n], [false, true, true, true]],
    [M.arrayfun("p", xs, p), [true, 2n], [1.0, 2.0]], [M.call("all", xs), [], true], [M.call("any", xs), [], false],
    [M.call("sum", xs), [], 0.0], [M.call("min", xs), [], []], [M.call("max", xs), [NaN, 2n], 2.0], [M.call("any", xs), [NaN], false],
    [M.call("all", xs), [NaN], true], [M.call("nnz", xs), [NaN, 0n], 1.0], [M.call("min", xs), [true, false], false],
    [M.call("sum", xs), [true, 2n], 3.0], [M.call("unique", xs), ["b", "a", "b"], ["a", "b"]], [M.call("unique", xs), [true, true], [true]],
    [M.call("unique", xs), [3n, 1n, 3n], [1.0, 3.0]], [M.call("numel", xs), 5n, 1.0], [M.index(xs, 1n), 5n, 5.0],
    [M.arrayfun("p", xs, M.binary("+", p, 1n)), 5n, [6.0]], [M.call("ismember", 1n, xs), [true], true], [M.call("numel", xs), "abc", 1.0],
    [M.call("numel", xs), new Map([["a", 1n]]), 1.0],
  ];
  const ERRORS: [any, unknown, Function, string][] = [
    [M.index(xs, 0n), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, 1.5), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, "a"), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, 5n), [1n, 2n, 3n, 2n], ValueError, "Index exceeds the number of array elements. Index must not exceed 4."],
    [M.arrayfun("p", xs, "a"), [1n, 2n], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."],
    [M.arrayfun("p", xs, M.binary("+", xs, 0n)), [1n, 2n], TypeError, "Operator '+' is not supported for operands of type 'array' and 'double'."],
    [M.arrayfun("p", xs, xs), [1n, 2n], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."],
    [M.call("sum", xs), ["a"], TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("all", xs), new Map([["a", 1n]]), TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("unique", xs), [new Map([["a", 1n]])], TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("ismember", xs, xs), [1n], TypeError, "ismember takes a scalar and an array of scalars."],
    [M.binary("==", xs, 1n), [1n], TypeError, "Operator '==' is not supported for operands of type 'array' and 'double'."],
    [M.binary("<", xs, 1n), new Map([["a", 1n]]), TypeError, "Operator '<' is not supported for operands of type 'struct' and 'double'."],
  ];
  for (const [expression, value, expected] of VALUES) {
    const result = MV.OfAny(expression, { xs: value });
    assert(alike(result, expected), `${MText.ToText(expression as any)}: ${String(result)}`);
  }
  for (const [expression, value, error, message] of ERRORS) raises(error, () => MV.OfAny(expression, { xs: value }), message);
  assert(Number.isNaN(MV.OfAny(M.call("max", xs), { xs: [NaN] })) && alike(MV.OfAny(M.call("unique", xs), { xs: [NaN, 1n, NaN] }), [1.0, NaN, NaN]));
  const Crew = new S.OfObject.Builder().ref().properties((q) => q.name("ranks").of(
    (t) => t.as_indexed((i) => i.of(new S.OfNative.Data(BigInt))))).create();
  Proxies.register("Crew", Crew);
  const crew = (Proxies.Builders as any).Crew().ranks([1n, 2n, 3n]).create(); // a list property is an array of doubles
  assert(alike(MV.OfAny(M.field(xs, "ranks"), { xs: crew }), [1.0, 2.0, 3.0]) && MV.OfAny(M.call("sum", M.field(xs, "ranks")), { xs: crew }) === 6.0);
}

## DIA-17 · Excel arrays: MAP and LAMBDA, AND and OR of any number of arguments, and the functions of arrays

In [ ]:
{
  const [xs, p] = [X.name("xs"), X.name("p")];
  assert(XText.ToText(X.function("AND", X.map_("p", X.field(xs, "pins"), X.infix(">", p, 0n)))) === "=AND(MAP(xs.pins, LAMBDA(p, p > 0)))");
  assert(XText.ToText(X.function("INDEX", xs, X.infix("+", p, 1n))) === "=INDEX(xs, p + 1)");
  assert(X.map_("p", xs, p).validate({ bound: ["xs"] }).length === 0
    && same([...SY.free(X.map_("p", xs, X.infix("+", p, X.name("q"))))].sort(), ["q", "xs"]));
  assert(X.function("AND", 1n, 2n, 3n).validate().length === 0 && same(X.function("ROWS", xs, 1n).validate({ bound: ["xs"] }), ["ROWS takes 1 arguments, got 2"]));
  assert(X.DIALECT.infer(X.function("AND", xs), { xs: D.Anything }).name() === "logical");
  assert(X.DIALECT.infer(X.map_("p", xs, p), { xs: D.Anything }) === D.Anything);
  assert(XV.OfAny(X.function("AND", true, true, false)) === false && XV.OfAny(X.function("OR", false, false, true)) === true);
  // IF takes 2 or 3 arguments, and AND, OR, SUM, MIN and MAX 1 to 255, as Excel's do.
  assert(X.function("IF", true, 1n).validate().length === 0 && XV.OfAny(X.function("IF", false, 1n)) === false && XV.OfAny(X.function("IF", true, 1n)) === 1.0);
  assert(same(X.function("IF", true).validate(), ["IF takes 2 or 3 arguments, got 1"]) && same(X.function("SUM").validate(), ["SUM takes 1 to 255 arguments, got 0"]));
  assert(XV.OfAny(X.function("SUM", 1n, xs, true), { xs: [2n, "a", true] }) === 4.0 && XV.OfAny(X.function("MAX", xs, 7n), { xs: [1n, 2n] }) === 7.0);
  assert(XV.OfAny(X.function("MIN", 3n, X.name("e")), { e: XD.Error.of("#N/A") }) === XD.Error.of("#N/A"));
  const arities: [unknown, string][] = [[X.function("IF", true, 1n, 2n, 3n), "IF takes 2 or 3 arguments, got 4"],
    [X.function("AND"), "AND takes 1 to 255 arguments, got 0"], [X.function("MIN"), "MIN takes 1 to 255 arguments, got 0"]];
  for (const [expression, message] of arities) raises(TypeError, () => XV.OfAny(expression), message);
  const VALUES: [any, unknown, unknown][] = [
    [X.function("AND", X.map_("p", xs, X.infix(">", p, 0n))), [1n, 2n, 3n, 2n], true],
    [X.function("OR", X.map_("p", xs, X.infix(">", p, 2n))), [1n, 2n, 3n, 2n], true],
    [X.function("SUM", X.map_("p", xs, X.function("IF", X.infix(">=", p, 2n), 1n, 0n))), [1n, 2n, 3n, 2n], 3.0],
    [X.function("ROWS", xs), [1n, 2n, 3n, 2n], 4.0], [X.function("INDEX", xs, 2n), [1n, 2n, 3n, 2n], 2.0],
    [X.function("INDEX", xs, 2.9), [1n, 2n, 3n, 2n], 2.0], [X.function("INDEX", xs, 5n), [1n, 2n], XD.Error.of("#REF!")],
    [X.function("INDEX", xs, 0n), [1n, 2n], XD.Error.of("#REF!")], [X.function("INDEX", xs, "x"), [1n, 2n], XD.Error.of("#VALUE!")],
    [X.function("MATCH", 3n, xs, 0n), [1n, 2n, 3n, 2n], 3.0], [X.function("MATCH", "B", xs, 0n), ["a", "b"], 2.0],
    [X.function("MATCH", 5n, xs, 0n), [1n, 2n], XD.Error.of("#N/A")], [X.function("MATCH", "1", xs, 0n), [1n, 2n], XD.Error.of("#N/A")],
    [X.function("MATCH", 1n, xs, 1n), [1n, 2n], XD.Error.of("#VALUE!")], [X.function("MATCH", true, xs, 0n), [1n, true], 2.0],
    [X.function("ISNUMBER", X.function("MATCH", 2n, xs, 0n)), [1n, 2n], true],
    [X.function("ISNUMBER", X.function("MATCH", 7n, xs, 0n)), [1n, 2n], false], [X.function("ISNUMBER", "1"), [], false],
    [X.function("SUM", xs), [1n, 2n, 3n, 2n], 8.0], [X.function("SUM", xs), [1n, "a", true, 2n], 3.0],
    [X.function("SUM", xs), [1n, XD.Error.of("#N/A")], XD.Error.of("#N/A")], [X.function("SUM", true), [], 1.0],
    [X.function("SUM", "2"), [], 2.0], [X.function("SUM", "a"), [], XD.Error.of("#VALUE!")], [X.function("MIN", xs), [3n, 1n, 2n], 1.0],
    [X.function("MAX", xs), [3n, 1n, 2n], 3.0], [X.function("MAX", xs), ["a"], 0.0], [X.function("MIN", xs), [], 0.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), [1n, 2n, 3n, 2n], 3.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), ["a", "A", 1n, true, "1"], 4.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), [XD.Error.of("#N/A"), XD.Error.of("#N/A"), new Map([["a", 1n]]), new Map([["a", 1n]])], 3.0],
    [X.function("ROWS", xs), 5n, 1.0], [X.function("INDEX", xs, 1n), 5n, 5.0],
    [X.function("ROWS", X.map_("p", xs, X.infix("*", p, 2n))), 5n, 1.0], [X.function("AND", xs), [true, "a", 1n], true],
    [X.function("AND", xs), ["a"], XD.Error.of("#VALUE!")], [X.function("OR", xs), [], XD.Error.of("#VALUE!")],
    [X.function("AND", xs, false), [true], false], [X.function("OR", xs), [0n, XD.Error.of("#DIV/0!")], XD.Error.of("#DIV/0!")],
    [X.function("INDEX", X.map_("p", xs, X.infix("-", p, "a")), 1n), [1n], XD.Error.of("#VALUE!")],
    [X.function("ROWS", X.map_("p", xs, p)), XD.Error.of("#REF!"), XD.Error.of("#REF!")], [X.infix("+", xs, 1n), [1n], XD.Error.of("#VALUE!")],
    [X.infix("=", xs, 1n), [1n], XD.Error.of("#VALUE!")],
    [X.function("MATCH", 1n, xs, "a"), [1n], XD.Error.of("#VALUE!")], [X.function("UNIQUE", xs), XD.Error.of("#REF!"), XD.Error.of("#REF!")],
  ];
  assert(XV.OfAny(X.function("INDEX", xs, X.name("n")), { xs: [1n], n: Infinity }) === XD.Error.of("#NUM!"));
  for (const [expression, value, expected] of VALUES) {
    const result = XV.OfAny(expression, { xs: value });
    assert(result === expected && typeof result === typeof expected, `${XText.ToText(expression as any)}: ${String(result)}`);
  }
  const Team = new S.OfObject.Builder().ref().properties((q) => q.name("ranks").of(
    (t) => t.as_indexed((i) => i.of(new S.OfNative.Data(BigInt))))).create();
  Proxies.register("Team", Team);
  const team = (Proxies.Builders as any).Team().ranks([1n, 2n, 3n]).create(); // a list property is an array of numbers
  assert(same(XV.OfAny(X.field(xs, "ranks"), { xs: team }) as unknown[], [1.0, 2.0, 3.0]) && XV.OfAny(X.function("SUM", X.field(xs, "ranks")), { xs: team }) === 6.0);
  assert(same(XV.OfAny(X.map_("p", xs, X.infix(">", p, 1n)), { xs: [1n, 2n] }) as unknown[], [false, true]));
}